# ANKA-50M — WorldPairs Controlled Proof Test

> **Status:** Controlled data-mechanism experiment

## Purpose

Test whether correct counterfactual WorldPairs improve relational accuracy relative to natural-only data and a matched wrong-pair control.

## Execution guidance

Generate and validate the paired corpus first, then train D0, D1, and W3 from matched initial weights and evaluate all three on the fixed ID/OOD suite.

## Primary outputs

Validated pair corpora, matched checkpoints, pair accuracy, natural loss, significance statistics, and a decision report.

## Reproducibility notes

- Code-cell outputs and execution counters are intentionally cleared in the public release.
- Seeds, persistent paths, and checkpoint-resume behavior remain in the code.
- Review path constants before execution; historical notebooks may reference earlier run directories.
- Execute sections from top to bottom unless a section explicitly describes an independent experiment arm.


## 1. Configure the controlled WorldPairs experiment

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 1 — Setup and experiment configuration
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "tokenizers>=0.15",
    "pandas>=2.0",
    "scipy>=1.10",
])

from google.colab import drive
drive.mount("/content/drive")

from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Optional
import gc
import hashlib
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import binomtest
from tokenizers import Tokenizer

SEED = 3407
EVAL_SEED = 93_411
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("In Colab select Runtime > Change runtime type > A100 GPU.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("this notebook BF16 capable CUDA GPU requires.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

DRIVE_RUN = Path("/content/drive/MyDrive/ANKA/run_01")
TOKENIZER_PATH = DRIVE_RUN / "tokenizer/tokenizer.json"
WORLD_ROOT = DRIVE_RUN / "worldpairs_proof_v1"
WORLD_ROOT.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 1_024
MICRO_BATCH = 64
TOKEN_BUDGET = 128_000_000
WORLD_RATIO = 0.10
USE_TORCH_COMPILE = False


def find_natural_manifest() -> Optional[Path]:
    preferred = [
        Path("/content/anka_shards/manifest.json"),
        DRIVE_RUN / "shards/manifest.json",
        DRIVE_RUN / "tokenized/manifest.json",
        DRIVE_RUN / "anka_shards/manifest.json",
    ]
    candidates = preferred + sorted(DRIVE_RUN.glob("**/manifest.json"))
    seen = set()
    for path in candidates:
        if path in seen or not path.exists():
            continue
        seen.add(path)
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(payload, dict) and isinstance(payload.get("shards"), list):
            return path
    return None


NATURAL_MANIFEST = find_natural_manifest()
assert TOKENIZER_PATH.exists(), f"Tokenizer not found: {TOKENIZER_PATH}"
assert NATURAL_MANIFEST is not None, (
    "Tokenized natural-data manifest not found. First expected location: "
    "/content/anka_shards/manifest.json"
)

print("GPU             :", torch.cuda.get_device_name(0))
print("VRAM            :", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")
print("Tokenizer       :", TOKENIZER_PATH)
print("Natural manifest  :", NATURAL_MANIFEST)
print("Results directory   :", WORLD_ROOT)
print("Budget / ratio    :", f"{TOKEN_BUDGET:,}", "/", f"{WORLD_RATIO:.0%}")
print("Sequence / batch:", SEQ_LEN, "/", MICRO_BATCH)


## 2. Define the WorldPairs relation schema

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 2 — WorldPairs schema
"""Canonical, immutable schema for ANKA-WorldPairs.

All world truth lives in ``WorldState.facts``. Convenience mappings such as
``held_by`` or ``contained_in`` must be derived from those facts rather than
stored as a second, potentially contradictory source of truth.
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum


class SchemaInvariantError(ValueError):
    """Raised when a symbolic world is internally inconsistent."""


class EntityType(str, Enum):
    PERSON = "person"
    OBJECT = "object"
    LOCATION = "location"
    CONTAINER = "container"
    SURFACE = "surface"


class Relation(str, Enum):
    # Binary spatial/possession predicates.
    AT = "at"
    HOLDING = "holding"
    INSIDE = "inside"
    ON_TOP_OF = "on_top_of"

    # Unary state and affordance predicates.
    OPEN = "open"
    CLOSED = "closed"
    SWITCHED_ON = "switched_on"
    SWITCHED_OFF = "switched_off"
    GRASPABLE = "graspable"
    HAND_EMPTY = "hand_empty"


BINARY_RELATIONS = frozenset(
    {Relation.AT, Relation.HOLDING, Relation.INSIDE, Relation.ON_TOP_OF}
)
UNARY_RELATIONS = frozenset(set(Relation) - BINARY_RELATIONS)
DIRECT_PLACEMENT_RELATIONS = frozenset(
    {Relation.AT, Relation.INSIDE, Relation.ON_TOP_OF}
)


@dataclass(frozen=True)
class Entity:
    entity_id: str
    name: str
    entity_type: EntityType

    def __post_init__(self) -> None:
        if not self.entity_id.strip():
            raise SchemaInvariantError("entity_id must not be empty")
        if not self.name.strip():
            raise SchemaInvariantError("entity name must not be empty")


@dataclass(frozen=True)
class Fact:
    relation: Relation
    subject_id: str
    object_id: str | None = None

    def __post_init__(self) -> None:
        if not self.subject_id.strip():
            raise SchemaInvariantError("fact subject_id must not be empty")
        if self.relation in BINARY_RELATIONS and self.object_id is None:
            raise SchemaInvariantError(
                f"binary relation {self.relation.value} requires object_id"
            )
        if self.relation in UNARY_RELATIONS and self.object_id is not None:
            raise SchemaInvariantError(
                f"unary relation {self.relation.value} forbids object_id"
            )


@dataclass(frozen=True)
class WorldState:
    """One immutable set of entities and ground predicates."""

    entities: tuple[Entity, ...]
    facts: frozenset[Fact]

    def __post_init__(self) -> None:
        entity_by_id = self.entity_by_id
        self._validate_unique_names()

        for fact in self.facts:
            if fact.subject_id not in entity_by_id:
                raise SchemaInvariantError(
                    f"unknown fact subject: {fact.subject_id}"
                )
            if fact.object_id is not None and fact.object_id not in entity_by_id:
                raise SchemaInvariantError(f"unknown fact object: {fact.object_id}")
            self._validate_fact_types(fact, entity_by_id)

        self._validate_exclusive_relations()
        self._validate_containment_graph()
        self._validate_opposite_unary_states()

    @property
    def entity_by_id(self) -> dict[str, Entity]:
        result: dict[str, Entity] = {}
        for entity in self.entities:
            if entity.entity_id in result:
                raise SchemaInvariantError(
                    f"duplicate entity_id: {entity.entity_id}"
                )
            result[entity.entity_id] = entity
        return result

    def _validate_unique_names(self) -> None:
        seen: dict[str, str] = {}
        for entity in self.entities:
            normalized = " ".join(entity.name.casefold().split())
            if normalized in seen:
                raise SchemaInvariantError(
                    "entity names must be unique within a world: "
                    f"{seen[normalized]} and {entity.entity_id}"
                )
            seen[normalized] = entity.entity_id

    @staticmethod
    def _validate_fact_types(
        fact: Fact, entity_by_id: dict[str, Entity]
    ) -> None:
        subject_type = entity_by_id[fact.subject_id].entity_type
        object_type = (
            entity_by_id[fact.object_id].entity_type
            if fact.object_id is not None
            else None
        )

        if fact.relation == Relation.HOLDING:
            if subject_type != EntityType.PERSON or object_type not in {
                EntityType.OBJECT,
                EntityType.CONTAINER,
            }:
                raise SchemaInvariantError(
                    "HOLDING requires person -> object/container"
                )
        elif fact.relation == Relation.AT:
            if subject_type == EntityType.LOCATION:
                raise SchemaInvariantError("a location cannot itself be AT a location")
            if object_type != EntityType.LOCATION:
                raise SchemaInvariantError("AT destination must be a location")
        elif fact.relation == Relation.INSIDE:
            if subject_type not in {EntityType.OBJECT, EntityType.CONTAINER}:
                raise SchemaInvariantError("INSIDE subject must be movable")
            if object_type != EntityType.CONTAINER:
                raise SchemaInvariantError("INSIDE destination must be a container")
        elif fact.relation == Relation.ON_TOP_OF:
            if subject_type not in {EntityType.OBJECT, EntityType.CONTAINER}:
                raise SchemaInvariantError("ON_TOP_OF subject must be movable")
            if object_type != EntityType.SURFACE:
                raise SchemaInvariantError("ON_TOP_OF destination must be a surface")
        elif fact.relation == Relation.HAND_EMPTY:
            if subject_type != EntityType.PERSON:
                raise SchemaInvariantError("HAND_EMPTY requires a person")
        elif fact.relation in {Relation.OPEN, Relation.CLOSED}:
            if subject_type != EntityType.CONTAINER:
                raise SchemaInvariantError("OPEN/CLOSED requires a container")
        elif fact.relation in {Relation.SWITCHED_ON, Relation.SWITCHED_OFF}:
            if subject_type != EntityType.OBJECT:
                raise SchemaInvariantError("SWITCHED_ON/OFF requires an object")
        elif fact.relation == Relation.GRASPABLE:
            if subject_type not in {EntityType.OBJECT, EntityType.CONTAINER}:
                raise SchemaInvariantError("GRASPABLE requires an object/container")

    def _validate_exclusive_relations(self) -> None:
        """Reject multiple direct placements or placement plus possession.

        Nested transport remains valid: INSIDE(key, pocket) may coexist with
        HOLDING(person, pocket). The key is indirectly transported with the
        pocket but is not directly held by the person.
        """

        direct_placements: dict[str, list[Fact]] = {}
        held_objects: dict[str, Fact] = {}

        for fact in self.facts:
            if fact.relation in DIRECT_PLACEMENT_RELATIONS:
                direct_placements.setdefault(fact.subject_id, []).append(fact)
            elif fact.relation == Relation.HOLDING:
                assert fact.object_id is not None
                if fact.object_id in held_objects:
                    raise SchemaInvariantError(
                        f"entity held by multiple people: {fact.object_id}"
                    )
                held_objects[fact.object_id] = fact

        for entity_id, placements in direct_placements.items():
            if len(placements) > 1:
                raise SchemaInvariantError(
                    f"entity has multiple direct placements: {entity_id}"
                )
            if entity_id in held_objects:
                raise SchemaInvariantError(
                    f"entity cannot be both held and directly placed: {entity_id}"
                )

        empty_hands = {
            fact.subject_id
            for fact in self.facts
            if fact.relation == Relation.HAND_EMPTY
        }
        active_holders = {
            fact.subject_id
            for fact in self.facts
            if fact.relation == Relation.HOLDING
        }
        contradictions = empty_hands & active_holders
        if contradictions:
            person_id = sorted(contradictions)[0]
            raise SchemaInvariantError(
                f"person cannot be HAND_EMPTY while HOLDING: {person_id}"
            )

    def _validate_containment_graph(self) -> None:
        parent = {
            fact.subject_id: fact.object_id
            for fact in self.facts
            if fact.relation == Relation.INSIDE
        }
        for start in parent:
            seen: set[str] = set()
            current: str | None = start
            while current in parent:
                if current in seen:
                    raise SchemaInvariantError(
                        f"containment cycle detected from: {start}"
                    )
                seen.add(current)
                current = parent[current]

    def _validate_opposite_unary_states(self) -> None:
        opposites = (
            (Relation.OPEN, Relation.CLOSED),
            (Relation.SWITCHED_ON, Relation.SWITCHED_OFF),
        )
        for entity in self.entities:
            for left, right in opposites:
                if Fact(left, entity.entity_id) in self.facts and Fact(
                    right, entity.entity_id
                ) in self.facts:
                    raise SchemaInvariantError(
                        f"opposite states on {entity.entity_id}: "
                        f"{left.value}/{right.value}"
                    )


class ActionType(str, Enum):
    MOVE_PERSON = "move_person"
    PICK_UP = "pick_up"
    PUT_IN = "put_in"
    PUT_ON = "put_on"
    OPEN = "open"
    CLOSE = "close"
    SWITCH_ON = "switch_on"
    SWITCH_OFF = "switch_off"


@dataclass(frozen=True)
class ActionInstance:
    action_type: ActionType
    arguments: tuple[str, ...]

    def __post_init__(self) -> None:
        expected_arity = {
            ActionType.MOVE_PERSON: 2,
            ActionType.PICK_UP: 2,
            ActionType.PUT_IN: 3,
            ActionType.PUT_ON: 3,
            ActionType.OPEN: 2,
            ActionType.CLOSE: 2,
            ActionType.SWITCH_ON: 2,
            ActionType.SWITCH_OFF: 2,
        }[self.action_type]
        if len(self.arguments) != expected_arity:
            raise SchemaInvariantError(
                f"{self.action_type.value} expects {expected_arity} arguments; "
                f"received {len(self.arguments)}"
            )
        if any(not value.strip() for value in self.arguments):
            raise SchemaInvariantError("action arguments must not be empty")


class QuestionType(str, Enum):
    LOCATION = "location"
    HOLDER = "holder"
    CONTAINER = "container"
    PROPERTY = "property"
    FEASIBILITY = "feasibility"


@dataclass(frozen=True)
class Question:
    question_type: QuestionType
    target_id: str
    relation: Relation | None = None

    def __post_init__(self) -> None:
        if not self.target_id.strip():
            raise SchemaInvariantError("question target_id must not be empty")
        if self.question_type == QuestionType.PROPERTY:
            if self.relation not in {
                Relation.OPEN,
                Relation.CLOSED,
                Relation.SWITCHED_ON,
                Relation.SWITCHED_OFF,
            }:
                raise SchemaInvariantError(
                    "PROPERTY question requires an open/closed or on/off relation"
                )
        elif self.relation is not None and self.question_type not in {
            QuestionType.CONTAINER,
            QuestionType.FEASIBILITY,
        }:
            raise SchemaInvariantError(
                f"{self.question_type.value} question does not accept relation"
            )


class ContrastKind(str, Enum):
    DESTINATION = "destination"
    HOLDER = "holder"
    CONTAINER = "container"
    PROPERTY_STATE = "property_state"
    EVENT_ORDER = "event_order"


@dataclass(frozen=True)
class Intervention:
    kind: ContrastKind
    event_index: int
    field_name: str
    old_value: str
    new_value: str

    def __post_init__(self) -> None:
        if self.event_index < 0:
            raise SchemaInvariantError("event_index must be non-negative")
        if not self.field_name.strip():
            raise SchemaInvariantError("field_name must not be empty")
        if self.old_value == self.new_value:
            raise SchemaInvariantError("intervention must change a value")


@dataclass(frozen=True)
class WorldExample:
    pair_family_id: str
    member_id: str
    initial_state: WorldState
    actions: tuple[ActionInstance, ...]
    question: Question
    answer_id: str
    contrast_kind: ContrastKind

    def __post_init__(self) -> None:
        if not self.pair_family_id.strip():
            raise SchemaInvariantError("pair_family_id must not be empty")
        if not self.member_id.strip():
            raise SchemaInvariantError("member_id must not be empty")
        if not self.actions:
            raise SchemaInvariantError("WorldExample requires at least one action")
        if not self.answer_id.strip():
            raise SchemaInvariantError("answer_id must not be empty")


@dataclass(frozen=True)
class WorldPair:
    factual: WorldExample
    counterfactual: WorldExample
    intervention: Intervention

    def __post_init__(self) -> None:
        if self.factual.pair_family_id != self.counterfactual.pair_family_id:
            raise SchemaInvariantError("pair members must share pair_family_id")
        if self.factual.member_id == self.counterfactual.member_id:
            raise SchemaInvariantError("pair members must have distinct member_id")
        if self.factual.contrast_kind != self.intervention.kind:
            raise SchemaInvariantError("factual contrast_kind mismatch")
        if self.counterfactual.contrast_kind != self.intervention.kind:
            raise SchemaInvariantError("counterfactual contrast_kind mismatch")


## 3. Define the deterministic world engine

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 3 — Deterministic world engine
"""Deterministic symbolic transition engine for ANKA-WorldPairs.

The engine never reads rendered text.  It consumes the canonical predicates in
``WorldState.facts`` and returns a new immutable ``WorldState`` after every
action.  Generation, rendering, and evaluation all call this module rather
than maintaining their own versions of world truth.
"""

from __future__ import annotations

from dataclasses import dataclass



class InvalidActionError(ValueError):
    """Raised when an action violates a world precondition."""


class UnanswerableQuestionError(ValueError):
    """Raised when a final state does not determine one unique answer."""


@dataclass(frozen=True)
class ExecutionResult:
    states: tuple[WorldState, ...]
    final_state: WorldState
    answer_id: str


def facts_with(
    state: WorldState,
    relation: Relation,
    *,
    subject_id: str | None = None,
    object_id: str | None = None,
) -> tuple[Fact, ...]:
    return tuple(
        sorted(
            (
                fact
                for fact in state.facts
                if fact.relation == relation
                and (subject_id is None or fact.subject_id == subject_id)
                and (object_id is None or fact.object_id == object_id)
            ),
            key=lambda fact: (fact.subject_id, fact.object_id or ""),
        )
    )


def _one_or_none(values: tuple[Fact, ...], description: str) -> Fact | None:
    if len(values) > 1:
        raise UnanswerableQuestionError(f"multiple {description} facts")
    return values[0] if values else None


def effective_location(state: WorldState, entity_id: str) -> str | None:
    """Resolve direct, held, contained, and supported locations recursively."""

    if entity_id not in state.entity_by_id:
        raise KeyError(f"unknown entity: {entity_id}")

    seen: set[str] = set()
    current = entity_id
    while True:
        if current in seen:
            raise UnanswerableQuestionError(
                f"location dependency cycle starting at {entity_id}"
            )
        seen.add(current)

        direct = _one_or_none(
            facts_with(state, Relation.AT, subject_id=current),
            f"AT({current}, ...)",
        )
        if direct is not None:
            return direct.object_id

        holders = facts_with(state, Relation.HOLDING, object_id=current)
        holder = _one_or_none(holders, f"HOLDING(..., {current})")
        if holder is not None:
            current = holder.subject_id
            continue

        inside = _one_or_none(
            facts_with(state, Relation.INSIDE, subject_id=current),
            f"INSIDE({current}, ...)",
        )
        if inside is not None:
            assert inside.object_id is not None
            current = inside.object_id
            continue

        supported = _one_or_none(
            facts_with(state, Relation.ON_TOP_OF, subject_id=current),
            f"ON_TOP_OF({current}, ...)",
        )
        if supported is not None:
            assert supported.object_id is not None
            current = supported.object_id
            continue

        return None


def _assert_type(
    state: WorldState, entity_id: str, allowed: set[EntityType], role: str
) -> None:
    entity = state.entity_by_id.get(entity_id)
    if entity is None:
        raise InvalidActionError(f"unknown {role}: {entity_id}")
    if entity.entity_type not in allowed:
        names = ", ".join(sorted(value.value for value in allowed))
        raise InvalidActionError(
            f"{role} {entity_id} must have type {{{names}}}; "
            f"found {entity.entity_type.value}"
        )


def _assert_colocated(state: WorldState, person_id: str, target_id: str) -> None:
    person_location = effective_location(state, person_id)
    target_location = effective_location(state, target_id)
    if person_location is None or target_location is None:
        raise InvalidActionError(
            f"cannot establish co-location for {person_id} and {target_id}"
        )
    if person_location != target_location:
        raise InvalidActionError(
            f"{person_id} and {target_id} are not co-located: "
            f"{person_location} != {target_location}"
        )


def _is_accessible(state: WorldState, entity_id: str) -> bool:
    """An item is inaccessible when nested in any closed container."""

    seen: set[str] = set()
    current = entity_id
    while True:
        if current in seen:
            return False
        seen.add(current)
        parent = _one_or_none(
            facts_with(state, Relation.INSIDE, subject_id=current),
            f"INSIDE({current}, ...)",
        )
        if parent is None:
            return True
        assert parent.object_id is not None
        if Fact(Relation.CLOSED, parent.object_id) in state.facts:
            return False
        current = parent.object_id


def _remove_relations(
    facts: set[Fact], *, subject_id: str, relations: set[Relation]
) -> None:
    facts.difference_update(
        fact
        for fact in tuple(facts)
        if fact.subject_id == subject_id and fact.relation in relations
    )


def _set_binary(
    facts: set[Fact], relation: Relation, subject_id: str, object_id: str
) -> None:
    _remove_relations(
        facts,
        subject_id=subject_id,
        relations={Relation.AT, Relation.INSIDE, Relation.ON_TOP_OF},
    )
    facts.difference_update(
        fact
        for fact in tuple(facts)
        if fact.relation == Relation.HOLDING and fact.object_id == subject_id
    )
    facts.add(Fact(relation, subject_id, object_id))


def _set_opposite(
    facts: set[Fact], subject_id: str, positive: Relation, negative: Relation
) -> None:
    facts.discard(Fact(positive, subject_id))
    facts.discard(Fact(negative, subject_id))
    facts.add(Fact(positive, subject_id))


def apply_action(state: WorldState, action: ActionInstance) -> WorldState:
    facts = set(state.facts)
    kind = action.action_type
    args = action.arguments

    if kind == ActionType.MOVE_PERSON:
        person_id, destination_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(state, destination_id, {EntityType.LOCATION}, "destination")
        _remove_relations(facts, subject_id=person_id, relations={Relation.AT})
        facts.add(Fact(Relation.AT, person_id, destination_id))

    elif kind == ActionType.PICK_UP:
        person_id, object_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(
            state,
            object_id,
            {EntityType.OBJECT, EntityType.CONTAINER},
            "picked entity",
        )
        if Fact(Relation.HAND_EMPTY, person_id) not in state.facts:
            raise InvalidActionError(f"{person_id} has no empty hand")
        if Fact(Relation.GRASPABLE, object_id) not in state.facts:
            raise InvalidActionError(f"{object_id} is not graspable")
        if not _is_accessible(state, object_id):
            raise InvalidActionError(f"{object_id} is inside a closed container")
        _assert_colocated(state, person_id, object_id)
        _remove_relations(
            facts,
            subject_id=object_id,
            relations={Relation.AT, Relation.INSIDE, Relation.ON_TOP_OF},
        )
        facts.add(Fact(Relation.HOLDING, person_id, object_id))
        facts.discard(Fact(Relation.HAND_EMPTY, person_id))

    elif kind == ActionType.PUT_IN:
        person_id, object_id, container_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(
            state,
            object_id,
            {EntityType.OBJECT, EntityType.CONTAINER},
            "placed entity",
        )
        _assert_type(state, container_id, {EntityType.CONTAINER}, "container")
        if object_id == container_id:
            raise InvalidActionError("an entity cannot be put inside itself")
        if Fact(Relation.HOLDING, person_id, object_id) not in state.facts:
            raise InvalidActionError(f"{person_id} is not holding {object_id}")
        if Fact(Relation.OPEN, container_id) not in state.facts:
            raise InvalidActionError(f"container {container_id} is not open")
        _assert_colocated(state, person_id, container_id)
        facts.discard(Fact(Relation.HOLDING, person_id, object_id))
        _set_binary(facts, Relation.INSIDE, object_id, container_id)
        if not facts_with(
            WorldState(state.entities, frozenset(facts)),
            Relation.HOLDING,
            subject_id=person_id,
        ):
            facts.add(Fact(Relation.HAND_EMPTY, person_id))

    elif kind == ActionType.PUT_ON:
        person_id, object_id, surface_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(
            state,
            object_id,
            {EntityType.OBJECT, EntityType.CONTAINER},
            "placed entity",
        )
        _assert_type(state, surface_id, {EntityType.SURFACE}, "surface")
        if Fact(Relation.HOLDING, person_id, object_id) not in state.facts:
            raise InvalidActionError(f"{person_id} is not holding {object_id}")
        _assert_colocated(state, person_id, surface_id)
        facts.discard(Fact(Relation.HOLDING, person_id, object_id))
        _set_binary(facts, Relation.ON_TOP_OF, object_id, surface_id)
        if not any(
            fact.relation == Relation.HOLDING and fact.subject_id == person_id
            for fact in facts
        ):
            facts.add(Fact(Relation.HAND_EMPTY, person_id))

    elif kind in {ActionType.OPEN, ActionType.CLOSE}:
        person_id, container_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(state, container_id, {EntityType.CONTAINER}, "container")
        _assert_colocated(state, person_id, container_id)
        desired = Relation.OPEN if kind == ActionType.OPEN else Relation.CLOSED
        opposite = Relation.CLOSED if desired == Relation.OPEN else Relation.OPEN
        _set_opposite(facts, container_id, desired, opposite)

    elif kind in {ActionType.SWITCH_ON, ActionType.SWITCH_OFF}:
        person_id, object_id = args
        _assert_type(state, person_id, {EntityType.PERSON}, "person")
        _assert_type(state, object_id, {EntityType.OBJECT}, "switchable object")
        _assert_colocated(state, person_id, object_id)
        desired = (
            Relation.SWITCHED_ON
            if kind == ActionType.SWITCH_ON
            else Relation.SWITCHED_OFF
        )
        opposite = (
            Relation.SWITCHED_OFF
            if desired == Relation.SWITCHED_ON
            else Relation.SWITCHED_ON
        )
        _set_opposite(facts, object_id, desired, opposite)

    else:  # pragma: no cover - enum exhaustiveness guard
        raise InvalidActionError(f"unsupported action: {kind}")

    return WorldState(entities=state.entities, facts=frozenset(facts))


def answer_question(state: WorldState, question: Question) -> str:
    target_id = question.target_id
    if target_id not in state.entity_by_id:
        raise UnanswerableQuestionError(f"unknown target: {target_id}")

    if question.question_type == QuestionType.LOCATION:
        result = effective_location(state, target_id)
        if result is None:
            raise UnanswerableQuestionError(f"no location for {target_id}")
        return result

    if question.question_type == QuestionType.HOLDER:
        fact = _one_or_none(
            facts_with(state, Relation.HOLDING, object_id=target_id),
            f"holder of {target_id}",
        )
        return fact.subject_id if fact is not None else "nobody"

    if question.question_type == QuestionType.CONTAINER:
        fact = _one_or_none(
            facts_with(state, Relation.INSIDE, subject_id=target_id),
            f"container of {target_id}",
        )
        return fact.object_id if fact is not None else "none"

    if question.question_type == QuestionType.PROPERTY:
        assert question.relation is not None
        if question.relation in {Relation.OPEN, Relation.CLOSED}:
            if Fact(Relation.OPEN, target_id) in state.facts:
                return "open"
            if Fact(Relation.CLOSED, target_id) in state.facts:
                return "closed"
        elif question.relation in {Relation.SWITCHED_ON, Relation.SWITCHED_OFF}:
            if Fact(Relation.SWITCHED_ON, target_id) in state.facts:
                return "on"
            if Fact(Relation.SWITCHED_OFF, target_id) in state.facts:
                return "off"
        raise UnanswerableQuestionError(
            f"no requested property state for {target_id}"
        )

    raise UnanswerableQuestionError(
        "FEASIBILITY questions require an explicit candidate action and are "
        "not part of WorldPairs-v1"
    )


def execute(
    initial_state: WorldState,
    actions: tuple[ActionInstance, ...],
    question: Question,
) -> ExecutionResult:
    states = [initial_state]
    current = initial_state
    for action in actions:
        current = apply_action(current, action)
        states.append(current)
    answer_id = answer_question(current, question)
    return ExecutionResult(tuple(states), current, answer_id)


## 4. Generate minimal counterfactual pairs

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 4 — Minimal counterfactual pair generator
"""Deterministic minimal-pair generator for ANKA-WorldPairs-v1."""

from __future__ import annotations

from dataclasses import dataclass
import hashlib
import random



TRAIN_NAMES = {
    "people": (
        "Alice", "Bob", "Carol", "David", "Emma", "Frank", "Grace",
        "Henry", "Iris", "Jack", "Karen", "Liam", "Maya", "Noah",
    ),
    "objects": (
        "key", "coin", "book", "apple", "ball", "card", "brush",
        "spoon", "toy", "note", "ring", "cup", "map", "pen",
    ),
    "containers": (
        "box", "drawer", "cabinet", "basket", "suitcase", "locker",
        "chest", "pouch", "crate", "case",
    ),
    "surfaces": ("table", "desk", "shelf", "bench", "counter"),
    "locations": (
        "kitchen", "garden", "office", "bedroom", "hallway", "garage",
        "library", "studio", "workshop", "lounge",
    ),
    "switchables": ("lamp", "radio", "fan", "heater", "speaker", "clock"),
}


EVAL_NAMES = {
    "people": (
        "Nora", "Owen", "Priya", "Quinn", "Rosa", "Simon", "Tara",
        "Victor", "Wendy", "Yusuf",
    ),
    "objects": (
        "badge", "ticket", "compass", "pencil", "bottle", "scarf",
        "wallet", "camera", "glove", "plate",
    ),
    "containers": (
        "trunk", "cupboard", "backpack", "carton", "bin", "folder",
        "satchel", "jar",
    ),
    "surfaces": ("stool", "platform", "stand", "worktop", "rack"),
    "locations": (
        "pantry", "courtyard", "attic", "basement", "balcony", "porch",
        "classroom", "warehouse", "foyer", "cafeteria",
    ),
    "switchables": ("projector", "monitor", "printer", "kettle", "torch"),
}


@dataclass(frozen=True)
class GeneratedPair:
    pair: WorldPair
    candidate_answer_ids: tuple[str, ...]
    split: str
    renderer_bank: str
    chain_depth: int
    reasoning_profile: str
    seed: int


def stable_seed(*parts: object) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def _sample_names(rng: random.Random, evaluation_names: bool) -> dict[str, tuple[str, ...]]:
    bank = EVAL_NAMES if evaluation_names else TRAIN_NAMES
    return {
        "people": tuple(rng.sample(bank["people"], 2)),
        "objects": tuple(rng.sample(bank["objects"], 1)),
        "containers": tuple(rng.sample(bank["containers"], 2)),
        "surfaces": tuple(rng.sample(bank["surfaces"], 1)),
        "locations": tuple(rng.sample(bank["locations"], 3)),
        "switchables": tuple(rng.sample(bank["switchables"], 1)),
    }


def _entities(names: dict[str, tuple[str, ...]]) -> tuple[Entity, ...]:
    return (
        Entity("person_a", names["people"][0], EntityType.PERSON),
        Entity("person_b", names["people"][1], EntityType.PERSON),
        Entity("target", names["objects"][0], EntityType.OBJECT),
        Entity("device", names["switchables"][0], EntityType.OBJECT),
        Entity("container_a", names["containers"][0], EntityType.CONTAINER),
        Entity("container_b", names["containers"][1], EntityType.CONTAINER),
        Entity("surface", names["surfaces"][0], EntityType.SURFACE),
        Entity("room_a", names["locations"][0], EntityType.LOCATION),
        Entity("room_b", names["locations"][1], EntityType.LOCATION),
        Entity("room_c", names["locations"][2], EntityType.LOCATION),
    )


def _base_facts(kind: ContrastKind) -> set[Fact]:
    facts = {
        Fact(Relation.AT, "person_a", "room_a"),
        Fact(Relation.AT, "person_b", "room_a"),
        Fact(Relation.AT, "device", "room_a"),
        Fact(Relation.AT, "container_a", "room_a"),
        Fact(Relation.AT, "container_b", "room_a"),
        Fact(Relation.AT, "surface", "room_a"),
        Fact(Relation.HAND_EMPTY, "person_a"),
        Fact(Relation.HAND_EMPTY, "person_b"),
        Fact(Relation.GRASPABLE, "target"),
        Fact(Relation.GRASPABLE, "container_a"),
        Fact(Relation.SWITCHED_OFF, "device"),
        Fact(Relation.OPEN, "container_a"),
        Fact(Relation.OPEN, "container_b"),
    }

    if kind == ContrastKind.CONTAINER:
        facts.add(Fact(Relation.HOLDING, "person_a", "target"))
        facts.discard(Fact(Relation.HAND_EMPTY, "person_a"))
    else:
        facts.add(Fact(Relation.AT, "target", "room_a"))

    if kind in {ContrastKind.PROPERTY_STATE, ContrastKind.EVENT_ORDER}:
        facts.discard(Fact(Relation.OPEN, "container_a"))
        facts.add(Fact(Relation.CLOSED, "container_a"))

    return facts


def _distractor_prefix(length: int) -> tuple[ActionInstance, ...]:
    result = []
    for index in range(length):
        action_type = (
            ActionType.SWITCH_ON if index % 2 == 0 else ActionType.SWITCH_OFF
        )
        result.append(ActionInstance(action_type, ("person_a", "device")))
    return tuple(result)


def _with_answer(
    family_id: str,
    member_id: str,
    state: WorldState,
    actions: tuple[ActionInstance, ...],
    question: Question,
    kind: ContrastKind,
) -> WorldExample:
    answer = execute(state, actions, question).answer_id
    return WorldExample(
        pair_family_id=family_id,
        member_id=member_id,
        initial_state=state,
        actions=actions,
        question=question,
        answer_id=answer,
        contrast_kind=kind,
    )


def generate_pair(
    *,
    index: int,
    kind: ContrastKind,
    split: str,
    seed: int = 3407,
    chain_depth: int | None = None,
) -> GeneratedPair:
    """Generate one deterministic, executable, answer-flipping pair.

    ``train`` and ``id`` use renderer bank ``train``. ``ood_template`` uses
    held-out surface realizations, ``ood_depth`` uses longer action chains,
    and ``ood_kind`` is reserved for the event-order contrast.
    """

    allowed_splits = {"train", "id", "ood_template", "ood_depth", "ood_kind"}
    if split not in allowed_splits:
        raise ValueError(f"unknown split: {split}")
    if index < 0:
        raise ValueError("index must be non-negative")
    if split == "ood_kind" and kind != ContrastKind.EVENT_ORDER:
        raise ValueError("ood_kind is reserved for EVENT_ORDER")
    if split == "train" and kind == ContrastKind.EVENT_ORDER:
        raise ValueError("EVENT_ORDER is held out from WorldPairs-v1 training")

    pair_seed = stable_seed("worldpairs-v1", seed, split, kind.value, index)
    rng = random.Random(pair_seed)
    evaluation_names = split in {"ood_template", "ood_depth", "ood_kind"}
    names = _sample_names(rng, evaluation_names)
    entities = _entities(names)
    state = WorldState(entities, frozenset(_base_facts(kind)))

    if split == "ood_depth" and kind not in {
        ContrastKind.DESTINATION,
        ContrastKind.HOLDER,
    }:
        raise ValueError(
            "ood_depth uses the two contrasts with genuine multi-step "
            "causal chains: DESTINATION or HOLDER"
        )

    if kind == ContrastKind.DESTINATION:
        if split == "ood_depth":
            reasoning_profile = "nested_transport"
            causal_prefix = (
                ActionInstance(ActionType.PICK_UP, ("person_a", "target")),
                ActionInstance(
                    ActionType.PUT_IN,
                    ("person_a", "target", "container_a"),
                ),
                ActionInstance(ActionType.PICK_UP, ("person_a", "container_a")),
            )
            causal_suffix: tuple[ActionInstance, ...] = ()
        else:
            reasoning_profile = rng.choice(("person_transport", "held_transport"))
            causal_prefix = (
                (ActionInstance(ActionType.PICK_UP, ("person_a", "target")),)
                if reasoning_profile == "held_transport"
                else ()
            )
            causal_suffix = ()
    elif kind == ContrastKind.HOLDER and split == "ood_depth":
        reasoning_profile = "holder_then_transport"
        causal_prefix = ()
        causal_suffix = (
            ActionInstance(ActionType.MOVE_PERSON, ("person_a", "room_b")),
        )
    elif kind == ContrastKind.HOLDER:
        reasoning_profile = "direct_holder"
        causal_prefix = ()
        causal_suffix = ()
    elif kind == ContrastKind.CONTAINER:
        reasoning_profile = "direct_container"
        causal_prefix = ()
        causal_suffix = ()
    elif kind == ContrastKind.PROPERTY_STATE:
        reasoning_profile = "state_transition"
        causal_prefix = ()
        causal_suffix = ()
    else:
        reasoning_profile = "ordered_state_transition"
        causal_prefix = ()
        causal_suffix = ()

    contrast_events = 2 if kind == ContrastKind.EVENT_ORDER else 1
    minimum_causal_depth = len(causal_prefix) + contrast_events + len(causal_suffix)
    min_depth = minimum_causal_depth
    if chain_depth is None:
        if split == "ood_depth":
            chain_depth = rng.randint(max(5, min_depth), 7)
        else:
            chain_depth = rng.randint(min_depth, 4)
    if chain_depth < min_depth:
        raise ValueError(
            f"{kind.value} requires chain_depth >= {min_depth}; got {chain_depth}"
        )

    distractor_count = chain_depth - minimum_causal_depth
    prefix = _distractor_prefix(distractor_count) + causal_prefix
    event_index = len(prefix)

    if kind == ContrastKind.DESTINATION:
        factual_tail = (ActionInstance(ActionType.MOVE_PERSON, ("person_a", "room_b")),)
        counter_tail = (ActionInstance(ActionType.MOVE_PERSON, ("person_a", "room_c")),)
        question_target = (
            "person_a" if reasoning_profile == "person_transport" else "target"
        )
        question = Question(QuestionType.LOCATION, question_target)
        intervention = Intervention(kind, event_index, "arguments[1]", "room_b", "room_c")
        candidates = ("room_b", "room_c")
    elif kind == ContrastKind.HOLDER:
        factual_tail = (ActionInstance(ActionType.PICK_UP, ("person_a", "target")),)
        counter_tail = (ActionInstance(ActionType.PICK_UP, ("person_b", "target")),)
        if reasoning_profile == "holder_then_transport":
            question = Question(QuestionType.LOCATION, "target")
            candidates = ("room_a", "room_b")
        else:
            question = Question(QuestionType.HOLDER, "target")
            candidates = ("person_a", "person_b")
        intervention = Intervention(kind, event_index, "arguments[0]", "person_a", "person_b")
    elif kind == ContrastKind.CONTAINER:
        factual_tail = (
            ActionInstance(ActionType.PUT_IN, ("person_a", "target", "container_a")),
        )
        counter_tail = (
            ActionInstance(ActionType.PUT_IN, ("person_a", "target", "container_b")),
        )
        question = Question(QuestionType.CONTAINER, "target", Relation.INSIDE)
        intervention = Intervention(
            kind, event_index, "arguments[2]", "container_a", "container_b"
        )
        candidates = ("container_a", "container_b")
    elif kind == ContrastKind.PROPERTY_STATE:
        factual_tail = (ActionInstance(ActionType.OPEN, ("person_a", "container_a")),)
        counter_tail = (ActionInstance(ActionType.CLOSE, ("person_a", "container_a")),)
        question = Question(QuestionType.PROPERTY, "container_a", Relation.OPEN)
        intervention = Intervention(kind, event_index, "action_type", "open", "close")
        candidates = ("open", "closed")
    elif kind == ContrastKind.EVENT_ORDER:
        open_event = ActionInstance(ActionType.OPEN, ("person_a", "container_a"))
        close_event = ActionInstance(ActionType.CLOSE, ("person_a", "container_a"))
        factual_tail = (open_event, close_event)
        counter_tail = (close_event, open_event)
        question = Question(QuestionType.PROPERTY, "container_a", Relation.OPEN)
        intervention = Intervention(
            kind, event_index, "event_order", "open>close", "close>open"
        )
        candidates = ("open", "closed")
    else:  # pragma: no cover
        raise AssertionError(kind)

    family_id = f"wp1-{split}-{kind.value}-{seed}-{index:09d}"
    factual_actions = prefix + factual_tail + causal_suffix
    counter_actions = prefix + counter_tail + causal_suffix
    factual = _with_answer(
        family_id, "A", state, factual_actions, question, kind
    )
    counterfactual = _with_answer(
        family_id, "B", state, counter_actions, question, kind
    )
    pair = WorldPair(factual, counterfactual, intervention)

    return GeneratedPair(
        pair=pair,
        candidate_answer_ids=candidates,
        split=split,
        renderer_bank="ood" if split == "ood_template" else "train",
        chain_depth=chain_depth,
        reasoning_profile=reasoning_profile,
        seed=pair_seed,
    )


## 5. Render pairs in controlled English

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 5 — Controlled-English renderer
"""Controlled-English renderer for verified WorldPairs examples.

The renderer is deliberately separate from ``parse.py``.  The parser uses its
own regular expressions, which lets round-trip tests catch renderer drift
instead of merely calling the renderer's inverse table.
"""

from __future__ import annotations

from dataclasses import dataclass
import hashlib



TRAIN_FACT_TEMPLATES = {
    Relation.AT: (
        "{s} is located in {o}.",
        "{s} starts in {o}.",
        "At first, {s} is in {o}.",
    ),
    Relation.HOLDING: (
        "{s} is holding {o}.",
        "{s} carries {o}.",
        "{o} is held by {s}.",
    ),
    Relation.INSIDE: (
        "{s} is inside {o}.",
        "{o} contains {s}.",
        "Initially, {s} rests inside {o}.",
    ),
    Relation.ON_TOP_OF: (
        "{s} is on top of {o}.",
        "{s} rests on {o}.",
        "On {o} sits {s}.",
    ),
    Relation.OPEN: ("{s} is open.", "The state of {s} is open.", "{s} starts open."),
    Relation.CLOSED: ("{s} is closed.", "The state of {s} is closed.", "{s} starts closed."),
    Relation.SWITCHED_ON: ("{s} is on.", "{s} is switched on.", "The device {s} starts on."),
    Relation.SWITCHED_OFF: ("{s} is off.", "{s} is switched off.", "The device {s} starts off."),
    Relation.GRASPABLE: ("{s} can be picked up.", "{s} is graspable.", "It is possible to pick up {s}."),
    Relation.HAND_EMPTY: ("{s} has an empty hand.", "{s}'s hand is empty.", "At first, {s} holds nothing."),
}


OOD_FACT_TEMPLATES = {
    Relation.AT: ("The location of {s} is {o}.", "You can find {s} in {o}."),
    Relation.HOLDING: ("In {s}'s hand is {o}.", "{s} has possession of {o}."),
    Relation.INSIDE: ("Within {o} is {s}.", "{s} has been placed within {o}."),
    Relation.ON_TOP_OF: ("{o} supports {s}.", "{s} lies upon {o}."),
    Relation.OPEN: ("{s} begins in the open state.", "Open is the initial state of {s}."),
    Relation.CLOSED: ("{s} begins in the closed state.", "Closed is the initial state of {s}."),
    Relation.SWITCHED_ON: ("Power to {s} is on.", "Initially, {s} is powered on."),
    Relation.SWITCHED_OFF: ("Power to {s} is off.", "Initially, {s} is powered off."),
    Relation.GRASPABLE: ("Someone can grasp {s}.", "Picking up {s} is possible."),
    Relation.HAND_EMPTY: ("{s} is not carrying anything.", "Nothing is in {s}'s hand."),
}


TRAIN_ACTION_TEMPLATES = {
    ActionType.MOVE_PERSON: ("{a0} moved to {a1}.", "{a0} went to {a1}.", "{a0} entered {a1}."),
    ActionType.PICK_UP: ("{a0} picked up {a1}.", "{a0} took {a1}.", "{a0} grasped {a1}."),
    ActionType.PUT_IN: ("{a0} put {a1} inside {a2}.", "{a0} placed {a1} into {a2}.", "{a0} stored {a1} in {a2}."),
    ActionType.PUT_ON: ("{a0} put {a1} on {a2}.", "{a0} placed {a1} atop {a2}.", "{a0} set {a1} on top of {a2}."),
    ActionType.OPEN: ("{a0} opened {a1}.", "{a0} made {a1} open.", "{a0} switched {a1} to the open state."),
    ActionType.CLOSE: ("{a0} closed {a1}.", "{a0} made {a1} closed.", "{a0} switched {a1} to the closed state."),
    ActionType.SWITCH_ON: ("{a0} switched on {a1}.", "{a0} turned {a1} on.", "{a0} powered on {a1}."),
    ActionType.SWITCH_OFF: ("{a0} switched off {a1}.", "{a0} turned {a1} off.", "{a0} powered off {a1}."),
}


OOD_ACTION_TEMPLATES = {
    ActionType.MOVE_PERSON: ("Next, {a0} travelled into {a1}.", "Afterward, {a0} relocated to {a1}."),
    ActionType.PICK_UP: ("Next, {a0} lifted {a1}.", "Afterward, {a0} took hold of {a1}."),
    ActionType.PUT_IN: ("Next, {a0} inserted {a1} into {a2}.", "Afterward, {a0} deposited {a1} within {a2}."),
    ActionType.PUT_ON: ("Next, {a0} laid {a1} upon {a2}.", "Afterward, {a0} positioned {a1} over {a2}."),
    ActionType.OPEN: ("Next, {a0} changed {a1} from closed to open.", "Afterward, {a0} unclosed {a1}."),
    ActionType.CLOSE: ("Next, {a0} changed {a1} from open to closed.", "Afterward, {a0} shut {a1}."),
    ActionType.SWITCH_ON: ("Next, {a0} activated {a1}.", "Afterward, {a0} supplied power to {a1}."),
    ActionType.SWITCH_OFF: ("Next, {a0} deactivated {a1}.", "Afterward, {a0} cut power to {a1}."),
}


TRAIN_QUESTION_TEMPLATES = {
    QuestionType.LOCATION: ("Where is {s} now?", "What is {s}'s current location?"),
    QuestionType.HOLDER: ("Who is holding {s}?", "Who has {s} now?"),
    QuestionType.CONTAINER: ("Which container holds {s}?", "What is {s} inside?"),
    QuestionType.PROPERTY: ("What is the current state of {s}?", "Is {s} open or closed now?"),
}


OOD_QUESTION_TEMPLATES = {
    QuestionType.LOCATION: ("Identify the place containing {s} now.", "In which location did {s} end up?"),
    QuestionType.HOLDER: ("Identify the person who possesses {s}.", "In whose hand did {s} end up?"),
    QuestionType.CONTAINER: ("Identify the container containing {s}.", "Within which container did {s} end up?"),
    QuestionType.PROPERTY: ("Report the final state of {s}.", "Did {s} end open or closed?"),
}


@dataclass(frozen=True)
class RenderedExample:
    prompt: str
    answer_text: str
    full_text: str


@dataclass(frozen=True)
class RenderedPair:
    factual: RenderedExample
    counterfactual: RenderedExample
    block_text: str


def _choice_index(seed: int, key: str, count: int) -> int:
    payload = f"{seed}|{key}".encode("utf-8")
    value = int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")
    return value % count


def _name(example: WorldExample, entity_id: str) -> str:
    if entity_id in {"open", "closed", "on", "off", "none", "nobody"}:
        return entity_id
    return example.initial_state.entity_by_id[entity_id].name


def _render_fact(example: WorldExample, fact: Fact, bank: str, seed: int, index: int) -> str:
    templates = TRAIN_FACT_TEMPLATES if bank == "train" else OOD_FACT_TEMPLATES
    options = templates[fact.relation]
    template = options[_choice_index(seed, f"fact:{index}:{fact.relation.value}", len(options))]
    values = {"s": _name(example, fact.subject_id)}
    if fact.object_id is not None:
        values["o"] = _name(example, fact.object_id)
    return template.format(**values)


def _render_action(example: WorldExample, action: ActionInstance, bank: str, seed: int, index: int) -> str:
    templates = TRAIN_ACTION_TEMPLATES if bank == "train" else OOD_ACTION_TEMPLATES
    options = templates[action.action_type]
    template = options[_choice_index(seed, f"action:{index}", len(options))]
    values = {f"a{i}": _name(example, value) for i, value in enumerate(action.arguments)}
    return template.format(**values)


def _render_question(example: WorldExample, bank: str, seed: int) -> str:
    templates = TRAIN_QUESTION_TEMPLATES if bank == "train" else OOD_QUESTION_TEMPLATES
    options = templates[example.question.question_type]
    template = options[_choice_index(seed, "question", len(options))]
    return template.format(s=_name(example, example.question.target_id))


def render_example(example: WorldExample, *, bank: str, seed: int) -> RenderedExample:
    if bank not in {"train", "ood"}:
        raise ValueError(f"unknown renderer bank: {bank}")
    facts = sorted(
        example.initial_state.facts,
        key=lambda fact: (fact.relation.value, fact.subject_id, fact.object_id or ""),
    )
    world_lines = [
        f"- {_render_fact(example, fact, bank, seed, index)}"
        for index, fact in enumerate(facts)
    ]
    action_lines = [
        f"{index + 1}. {_render_action(example, action, bank, seed, index)}"
        for index, action in enumerate(example.actions)
    ]
    question = _render_question(example, bank, seed)
    answer = _name(example, example.answer_id)
    prompt = "\n".join(
        ["World:", *world_lines, "Events:", *action_lines, f"Question: {question}", "Answer:"]
    )
    full_text = f"{prompt} {answer}."
    return RenderedExample(prompt=prompt, answer_text=answer, full_text=full_text)


def render_pair(generated: GeneratedPair) -> RenderedPair:
    # The same seed/template plan is used for both members.  Consequently a
    # changed argument changes only the entity word, not unrelated phrasing.
    factual = render_example(
        generated.pair.factual,
        bank=generated.renderer_bank,
        seed=generated.seed,
    )
    counterfactual = render_example(
        generated.pair.counterfactual,
        bank=generated.renderer_bank,
        seed=generated.seed,
    )
    block = (
        "Paired world A\n"
        + factual.full_text
        + "\n\nPaired world B\n"
        + counterfactual.full_text
    )
    return RenderedPair(factual, counterfactual, block)


## 6. Parse generated examples independently

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 6 — Independent parser
"""Independent parser for the controlled-English WorldPairs renderer."""

from __future__ import annotations

from dataclasses import dataclass
import re



class ParseError(ValueError):
    pass


@dataclass(frozen=True)
class ParsedExample:
    initial_state: WorldState
    actions: tuple[ActionInstance, ...]
    question: Question
    answer_id: str


def _name_pattern(entities: tuple[Entity, ...]) -> str:
    names = sorted((re.escape(entity.name) for entity in entities), key=len, reverse=True)
    return "(?:" + "|".join(names) + ")"


def _normalize_entity_names(
    text: str, entities: tuple[Entity, ...]
) -> tuple[str, tuple[Entity, ...]]:
    """Replace surface names with stable ID markers before regex parsing.

    Generated worlds reuse the same entity IDs even when their English names
    change.  This makes the regex patterns cacheable across millions of
    examples while the replacement step still verifies every actual name.
    """

    by_name = {entity.name: f"§{entity.entity_id}§" for entity in entities}
    alternatives = "|".join(
        sorted((re.escape(name) for name in by_name), key=len, reverse=True)
    )
    pattern = re.compile(rf"(?<!\w)(?:{alternatives})(?!\w)")
    normalized = pattern.sub(lambda match: by_name[match.group(0)], text)
    normalized_entities = tuple(
        Entity(entity.entity_id, f"§{entity.entity_id}§", entity.entity_type)
        for entity in entities
    )
    return normalized, normalized_entities


def _id_map(entities: tuple[Entity, ...]) -> dict[str, str]:
    return {entity.name.casefold(): entity.entity_id for entity in entities}


def _entity_id(name: str, mapping: dict[str, str]) -> str:
    try:
        return mapping[name.casefold()]
    except KeyError as error:
        raise ParseError(f"unknown rendered entity name: {name}") from error


def _fullmatch(pattern: str, text: str, name_pattern: str):
    return re.fullmatch(pattern.replace("{N}", name_pattern), text)


def _parse_fact(text: str, entities: tuple[Entity, ...]) -> Fact:
    n = _name_pattern(entities)
    ids = _id_map(entities)

    binary_patterns: tuple[tuple[Relation, str], ...] = (
        (Relation.AT, r"(?P<s>{N}) is located in (?P<o>{N})\."),
        (Relation.AT, r"(?P<s>{N}) starts in (?P<o>{N})\."),
        (Relation.AT, r"At first, (?P<s>{N}) is in (?P<o>{N})\."),
        (Relation.AT, r"The location of (?P<s>{N}) is (?P<o>{N})\."),
        (Relation.AT, r"You can find (?P<s>{N}) in (?P<o>{N})\."),
        (Relation.HOLDING, r"(?P<s>{N}) is holding (?P<o>{N})\."),
        (Relation.HOLDING, r"(?P<s>{N}) carries (?P<o>{N})\."),
        (Relation.HOLDING, r"(?P<o>{N}) is held by (?P<s>{N})\."),
        (Relation.HOLDING, r"In (?P<s>{N})'s hand is (?P<o>{N})\."),
        (Relation.HOLDING, r"(?P<s>{N}) has possession of (?P<o>{N})\."),
        (Relation.INSIDE, r"(?P<s>{N}) is inside (?P<o>{N})\."),
        (Relation.INSIDE, r"(?P<o>{N}) contains (?P<s>{N})\."),
        (Relation.INSIDE, r"Initially, (?P<s>{N}) rests inside (?P<o>{N})\."),
        (Relation.INSIDE, r"Within (?P<o>{N}) is (?P<s>{N})\."),
        (Relation.INSIDE, r"(?P<s>{N}) has been placed within (?P<o>{N})\."),
        (Relation.ON_TOP_OF, r"(?P<s>{N}) is on top of (?P<o>{N})\."),
        (Relation.ON_TOP_OF, r"(?P<s>{N}) rests on (?P<o>{N})\."),
        (Relation.ON_TOP_OF, r"On (?P<o>{N}) sits (?P<s>{N})\."),
        (Relation.ON_TOP_OF, r"(?P<o>{N}) supports (?P<s>{N})\."),
        (Relation.ON_TOP_OF, r"(?P<s>{N}) lies upon (?P<o>{N})\."),
    )
    for relation, pattern in binary_patterns:
        match = _fullmatch(pattern, text, n)
        if match:
            return Fact(
                relation,
                _entity_id(match.group("s"), ids),
                _entity_id(match.group("o"), ids),
            )

    unary_patterns: tuple[tuple[Relation, str], ...] = (
        (Relation.OPEN, r"(?P<s>{N}) is open\."),
        (Relation.OPEN, r"The state of (?P<s>{N}) is open\."),
        (Relation.OPEN, r"(?P<s>{N}) starts open\."),
        (Relation.OPEN, r"(?P<s>{N}) begins in the open state\."),
        (Relation.OPEN, r"Open is the initial state of (?P<s>{N})\."),
        (Relation.CLOSED, r"(?P<s>{N}) is closed\."),
        (Relation.CLOSED, r"The state of (?P<s>{N}) is closed\."),
        (Relation.CLOSED, r"(?P<s>{N}) starts closed\."),
        (Relation.CLOSED, r"(?P<s>{N}) begins in the closed state\."),
        (Relation.CLOSED, r"Closed is the initial state of (?P<s>{N})\."),
        (Relation.SWITCHED_ON, r"(?P<s>{N}) is on\."),
        (Relation.SWITCHED_ON, r"(?P<s>{N}) is switched on\."),
        (Relation.SWITCHED_ON, r"The device (?P<s>{N}) starts on\."),
        (Relation.SWITCHED_ON, r"Power to (?P<s>{N}) is on\."),
        (Relation.SWITCHED_ON, r"Initially, (?P<s>{N}) is powered on\."),
        (Relation.SWITCHED_OFF, r"(?P<s>{N}) is off\."),
        (Relation.SWITCHED_OFF, r"(?P<s>{N}) is switched off\."),
        (Relation.SWITCHED_OFF, r"The device (?P<s>{N}) starts off\."),
        (Relation.SWITCHED_OFF, r"Power to (?P<s>{N}) is off\."),
        (Relation.SWITCHED_OFF, r"Initially, (?P<s>{N}) is powered off\."),
        (Relation.GRASPABLE, r"(?P<s>{N}) can be picked up\."),
        (Relation.GRASPABLE, r"(?P<s>{N}) is graspable\."),
        (Relation.GRASPABLE, r"It is possible to pick up (?P<s>{N})\."),
        (Relation.GRASPABLE, r"Someone can grasp (?P<s>{N})\."),
        (Relation.GRASPABLE, r"Picking up (?P<s>{N}) is possible\."),
        (Relation.HAND_EMPTY, r"(?P<s>{N}) has an empty hand\."),
        (Relation.HAND_EMPTY, r"(?P<s>{N})'s hand is empty\."),
        (Relation.HAND_EMPTY, r"At first, (?P<s>{N}) holds nothing\."),
        (Relation.HAND_EMPTY, r"(?P<s>{N}) is not carrying anything\."),
        (Relation.HAND_EMPTY, r"Nothing is in (?P<s>{N})'s hand\."),
    )
    for relation, pattern in unary_patterns:
        match = _fullmatch(pattern, text, n)
        if match:
            return Fact(relation, _entity_id(match.group("s"), ids))
    raise ParseError(f"unrecognized fact sentence: {text!r}")


def _parse_action(text: str, entities: tuple[Entity, ...]) -> ActionInstance:
    n = _name_pattern(entities)
    ids = _id_map(entities)
    patterns: tuple[tuple[ActionType, str, tuple[str, ...]], ...] = (
        (ActionType.MOVE_PERSON, r"(?P<a0>{N}) moved to (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.MOVE_PERSON, r"(?P<a0>{N}) went to (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.MOVE_PERSON, r"(?P<a0>{N}) entered (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.MOVE_PERSON, r"Next, (?P<a0>{N}) travelled into (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.MOVE_PERSON, r"Afterward, (?P<a0>{N}) relocated to (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PICK_UP, r"(?P<a0>{N}) picked up (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PICK_UP, r"(?P<a0>{N}) took (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PICK_UP, r"(?P<a0>{N}) grasped (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PICK_UP, r"Next, (?P<a0>{N}) lifted (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PICK_UP, r"Afterward, (?P<a0>{N}) took hold of (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.PUT_IN, r"(?P<a0>{N}) put (?P<a1>{N}) inside (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_IN, r"(?P<a0>{N}) placed (?P<a1>{N}) into (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_IN, r"(?P<a0>{N}) stored (?P<a1>{N}) in (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_IN, r"Next, (?P<a0>{N}) inserted (?P<a1>{N}) into (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_IN, r"Afterward, (?P<a0>{N}) deposited (?P<a1>{N}) within (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_ON, r"(?P<a0>{N}) put (?P<a1>{N}) on (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_ON, r"(?P<a0>{N}) placed (?P<a1>{N}) atop (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_ON, r"(?P<a0>{N}) set (?P<a1>{N}) on top of (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_ON, r"Next, (?P<a0>{N}) laid (?P<a1>{N}) upon (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.PUT_ON, r"Afterward, (?P<a0>{N}) positioned (?P<a1>{N}) over (?P<a2>{N})\.", ("a0", "a1", "a2")),
        (ActionType.OPEN, r"(?P<a0>{N}) opened (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.OPEN, r"(?P<a0>{N}) made (?P<a1>{N}) open\.", ("a0", "a1")),
        (ActionType.OPEN, r"(?P<a0>{N}) switched (?P<a1>{N}) to the open state\.", ("a0", "a1")),
        (ActionType.OPEN, r"Next, (?P<a0>{N}) changed (?P<a1>{N}) from closed to open\.", ("a0", "a1")),
        (ActionType.OPEN, r"Afterward, (?P<a0>{N}) unclosed (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.CLOSE, r"(?P<a0>{N}) closed (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.CLOSE, r"(?P<a0>{N}) made (?P<a1>{N}) closed\.", ("a0", "a1")),
        (ActionType.CLOSE, r"(?P<a0>{N}) switched (?P<a1>{N}) to the closed state\.", ("a0", "a1")),
        (ActionType.CLOSE, r"Next, (?P<a0>{N}) changed (?P<a1>{N}) from open to closed\.", ("a0", "a1")),
        (ActionType.CLOSE, r"Afterward, (?P<a0>{N}) shut (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_ON, r"(?P<a0>{N}) switched on (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_ON, r"(?P<a0>{N}) turned (?P<a1>{N}) on\.", ("a0", "a1")),
        (ActionType.SWITCH_ON, r"(?P<a0>{N}) powered on (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_ON, r"Next, (?P<a0>{N}) activated (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_ON, r"Afterward, (?P<a0>{N}) supplied power to (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_OFF, r"(?P<a0>{N}) switched off (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_OFF, r"(?P<a0>{N}) turned (?P<a1>{N}) off\.", ("a0", "a1")),
        (ActionType.SWITCH_OFF, r"(?P<a0>{N}) powered off (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_OFF, r"Next, (?P<a0>{N}) deactivated (?P<a1>{N})\.", ("a0", "a1")),
        (ActionType.SWITCH_OFF, r"Afterward, (?P<a0>{N}) cut power to (?P<a1>{N})\.", ("a0", "a1")),
    )
    for action_type, pattern, groups in patterns:
        match = _fullmatch(pattern, text, n)
        if match:
            return ActionInstance(
                action_type,
                tuple(_entity_id(match.group(group), ids) for group in groups),
            )
    raise ParseError(f"unrecognized action sentence: {text!r}")


def _parse_question(text: str, entities: tuple[Entity, ...]) -> Question:
    n = _name_pattern(entities)
    ids = _id_map(entities)
    patterns: tuple[tuple[QuestionType, Relation | None, str], ...] = (
        (QuestionType.LOCATION, None, r"Where is (?P<s>{N}) now\?"),
        (QuestionType.LOCATION, None, r"What is (?P<s>{N})'s current location\?"),
        (QuestionType.LOCATION, None, r"Identify the place containing (?P<s>{N}) now\."),
        (QuestionType.LOCATION, None, r"In which location did (?P<s>{N}) end up\?"),
        (QuestionType.HOLDER, None, r"Who is holding (?P<s>{N})\?"),
        (QuestionType.HOLDER, None, r"Who has (?P<s>{N}) now\?"),
        (QuestionType.HOLDER, None, r"Identify the person who possesses (?P<s>{N})\."),
        (QuestionType.HOLDER, None, r"In whose hand did (?P<s>{N}) end up\?"),
        (QuestionType.CONTAINER, Relation.INSIDE, r"Which container holds (?P<s>{N})\?"),
        (QuestionType.CONTAINER, Relation.INSIDE, r"What is (?P<s>{N}) inside\?"),
        (QuestionType.CONTAINER, Relation.INSIDE, r"Identify the container containing (?P<s>{N})\."),
        (QuestionType.CONTAINER, Relation.INSIDE, r"Within which container did (?P<s>{N}) end up\?"),
        (QuestionType.PROPERTY, Relation.OPEN, r"What is the current state of (?P<s>{N})\?"),
        (QuestionType.PROPERTY, Relation.OPEN, r"Is (?P<s>{N}) open or closed now\?"),
        (QuestionType.PROPERTY, Relation.OPEN, r"Report the final state of (?P<s>{N})\."),
        (QuestionType.PROPERTY, Relation.OPEN, r"Did (?P<s>{N}) end open or closed\?"),
    )
    for question_type, relation, pattern in patterns:
        match = _fullmatch(pattern, text, n)
        if match:
            return Question(question_type, _entity_id(match.group("s"), ids), relation)
    raise ParseError(f"unrecognized question: {text!r}")


def parse_example(text: str, entities: tuple[Entity, ...]) -> ParsedExample:
    original_entities = entities
    text, entities = _normalize_entity_names(text, entities)
    lines = text.splitlines()
    if not lines or lines[0] != "World:":
        raise ParseError("example must start with 'World:'")
    try:
        events_index = lines.index("Events:")
    except ValueError as error:
        raise ParseError("missing Events section") from error

    world_lines = lines[1:events_index]
    question_indices = [i for i, line in enumerate(lines) if line.startswith("Question: ")]
    answer_indices = [i for i, line in enumerate(lines) if line.startswith("Answer:")]
    if len(question_indices) != 1 or len(answer_indices) != 1:
        raise ParseError("expected one Question and one Answer line")
    question_index = question_indices[0]
    answer_index = answer_indices[0]
    if not events_index < question_index < answer_index:
        raise ParseError("invalid section order")

    facts = []
    for line in world_lines:
        if not line.startswith("- "):
            raise ParseError(f"invalid fact line: {line!r}")
        facts.append(_parse_fact(line[2:], entities))

    actions = []
    for expected, line in enumerate(lines[events_index + 1:question_index], start=1):
        prefix = f"{expected}. "
        if not line.startswith(prefix):
            raise ParseError(f"invalid action numbering: {line!r}")
        actions.append(_parse_action(line[len(prefix):], entities))

    question = _parse_question(lines[question_index][len("Question: "):], entities)
    answer_text = lines[answer_index][len("Answer:"):].strip()
    if answer_text.endswith("."):
        answer_text = answer_text[:-1]
    mapping = _id_map(entities)
    answer_id = mapping.get(answer_text.casefold(), answer_text.casefold())
    return ParsedExample(
        initial_state=WorldState(original_entities, frozenset(facts)),
        actions=tuple(actions),
        question=question,
        answer_id=answer_id,
    )


## 7. Validate semantic consistency

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 7 — Validator
"""Strict structural, execution, rendering, and leakage checks."""

from __future__ import annotations

from dataclasses import dataclass
from itertools import zip_longest



class PairValidationError(AssertionError):
    pass


@dataclass(frozen=True)
class ValidationReport:
    family_id: str
    contrast_kind: str
    answer_a: str
    answer_b: str
    action_differences: tuple[int, ...]
    rendered_line_differences: tuple[int, ...]
    parser_roundtrip: bool


def _action_differences(
    left: tuple[ActionInstance, ...], right: tuple[ActionInstance, ...]
) -> tuple[int, ...]:
    if len(left) != len(right):
        raise PairValidationError("pair action sequences have different lengths")
    return tuple(index for index, values in enumerate(zip(left, right)) if values[0] != values[1])


def _validate_intervention(generated: GeneratedPair) -> tuple[int, ...]:
    pair = generated.pair
    left = pair.factual.actions
    right = pair.counterfactual.actions
    intervention = pair.intervention
    differences = _action_differences(left, right)

    if intervention.kind == ContrastKind.EVENT_ORDER:
        expected = (intervention.event_index, intervention.event_index + 1)
        if differences != expected:
            raise PairValidationError(
                f"event-order pair differs at {differences}; expected {expected}"
            )
        if not (left[expected[0]] == right[expected[1]] and left[expected[1]] == right[expected[0]]):
            raise PairValidationError("event-order pair is not an exact adjacent swap")
        return differences

    expected = (intervention.event_index,)
    if differences != expected:
        raise PairValidationError(
            f"minimal pair differs at {differences}; expected only {expected}"
        )
    old_action = left[intervention.event_index]
    new_action = right[intervention.event_index]
    if intervention.field_name == "action_type":
        if old_action.arguments != new_action.arguments:
            raise PairValidationError("action-type intervention also changed arguments")
        if old_action.action_type.value != intervention.old_value:
            raise PairValidationError("intervention old action type is inaccurate")
        if new_action.action_type.value != intervention.new_value:
            raise PairValidationError("intervention new action type is inaccurate")
    elif intervention.field_name.startswith("arguments["):
        argument_index = int(intervention.field_name[10:-1])
        if old_action.action_type != new_action.action_type:
            raise PairValidationError("argument intervention changed action type")
        if len(old_action.arguments) != len(new_action.arguments):
            raise PairValidationError("argument intervention changed arity")
        other_indices = set(range(len(old_action.arguments))) - {argument_index}
        if any(old_action.arguments[i] != new_action.arguments[i] for i in other_indices):
            raise PairValidationError("argument intervention changed multiple fields")
        if old_action.arguments[argument_index] != intervention.old_value:
            raise PairValidationError("intervention old argument is inaccurate")
        if new_action.arguments[argument_index] != intervention.new_value:
            raise PairValidationError("intervention new argument is inaccurate")
    else:
        raise PairValidationError(
            f"unsupported intervention field: {intervention.field_name}"
        )
    return differences


def _validate_example(example: WorldExample) -> str:
    result = execute(example.initial_state, example.actions, example.question)
    if result.answer_id != example.answer_id:
        raise PairValidationError(
            f"declared answer {example.answer_id!r} != engine {result.answer_id!r}"
        )
    return result.answer_id


def _validate_roundtrip(example: WorldExample, rendered_text: str) -> None:
    parsed = parse_example(rendered_text, example.initial_state.entities)
    if parsed.initial_state != example.initial_state:
        raise PairValidationError("parser round-trip changed initial state")
    if parsed.actions != example.actions:
        raise PairValidationError("parser round-trip changed actions")
    if parsed.question != example.question:
        raise PairValidationError("parser round-trip changed question")
    if parsed.answer_id != example.answer_id:
        raise PairValidationError("parser round-trip changed answer")
    replayed = execute(parsed.initial_state, parsed.actions, parsed.question)
    if replayed.answer_id != parsed.answer_id:
        raise PairValidationError("parsed text fails independent engine replay")


def validate_pair(
    generated: GeneratedPair,
    rendered: RenderedPair | None = None,
) -> ValidationReport:
    pair = generated.pair
    if pair.factual.initial_state != pair.counterfactual.initial_state:
        raise PairValidationError("pair members must share the exact initial state")
    if pair.factual.question != pair.counterfactual.question:
        raise PairValidationError("pair members must share the exact question")
    if pair.factual.contrast_kind != pair.counterfactual.contrast_kind:
        raise PairValidationError("pair members have different contrast kinds")

    differences = _validate_intervention(generated)
    answer_a = _validate_example(pair.factual)
    answer_b = _validate_example(pair.counterfactual)
    if answer_a == answer_b:
        raise PairValidationError("intervention did not flip the answer")
    candidates = set(generated.candidate_answer_ids)
    if answer_a not in candidates or answer_b not in candidates:
        raise PairValidationError("correct pair answers are missing from candidates")

    rendered = rendered if rendered is not None else render_pair(generated)
    _validate_roundtrip(pair.factual, rendered.factual.full_text)
    _validate_roundtrip(pair.counterfactual, rendered.counterfactual.full_text)

    left_lines = rendered.factual.full_text.splitlines()
    right_lines = rendered.counterfactual.full_text.splitlines()
    line_differences = tuple(
        index
        for index, (left, right) in enumerate(
            zip_longest(left_lines, right_lines, fillvalue=None)
        )
        if left != right
    )
    expected_different_lines = len(differences) + 1  # actions + answer
    if len(line_differences) != expected_different_lines:
        raise PairValidationError(
            "rendered members changed unrelated text: "
            f"{len(line_differences)} differing lines; "
            f"expected {expected_different_lines}"
        )

    return ValidationReport(
        family_id=pair.factual.pair_family_id,
        contrast_kind=pair.intervention.kind.value,
        answer_a=answer_a,
        answer_b=answer_b,
        action_differences=differences,
        rendered_line_differences=line_differences,
        parser_roundtrip=True,
    )


def assert_disjoint_families(groups: dict[str, list[GeneratedPair]]) -> None:
    seen: dict[str, str] = {}
    for split, items in groups.items():
        for item in items:
            family_id = item.pair.factual.pair_family_id
            previous = seen.get(family_id)
            if previous is not None:
                raise PairValidationError(
                    f"family leakage: {family_id} occurs in {previous} and {split}"
                )
            seen[family_id] = split


## 8. Build the fixed packed binary corpus

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 8 — Fixed binary corpus builder
"""Build reusable fixed-row WorldPairs corpora and matched controls.

Two corpora are produced from exactly the same rendered individual examples:

* ``paired`` keeps A_i next to its true counterfactual B_i.
* ``matched_unpaired`` keeps A_i but rotates B across families inside a chunk.

Thus vocabulary, templates, answer frequency, and synthetic-token volume are
matched.  Only the causal pairing signal differs.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path
import random
from typing import Iterable, Protocol

import numpy as np



GENERATOR_VERSION = "worldpairs-v1.0"
TRAIN_CONTRASTS = (
    ContrastKind.DESTINATION,
    ContrastKind.HOLDER,
    ContrastKind.CONTAINER,
    ContrastKind.PROPERTY_STATE,
)


class Encoding(Protocol):
    ids: list[int]


class BatchTokenizer(Protocol):
    def encode_batch(self, inputs: list[str], add_special_tokens: bool = False) -> list[Encoding]: ...


@dataclass(frozen=True)
class PackedCorpusStats:
    rows: int
    valid_tokens: int
    predicted_tokens: int
    padded_tokens: int
    utilization: float
    maximum_block_tokens: int
    mean_block_tokens: float
    sha256_tokens: str
    sha256_lengths: str


def _pair_block(left: RenderedExample, right: RenderedExample) -> str:
    return (
        "Paired world A\n"
        + left.full_text
        + "\n\nPaired world B\n"
        + right.full_text
    )


def _balanced_chunk(
    *,
    start_by_kind: dict[ContrastKind, int],
    pair_count: int,
    seed: int,
) -> list[GeneratedPair]:
    rng = random.Random(seed)
    result: list[GeneratedPair] = []
    while len(result) < pair_count:
        order = list(TRAIN_CONTRASTS)
        rng.shuffle(order)
        for kind in order:
            if len(result) >= pair_count:
                break
            index = start_by_kind[kind]
            start_by_kind[kind] += 1
            result.append(
                generate_pair(
                    index=index,
                    kind=kind,
                    split="train",
                    seed=seed,
                )
            )
    return result


def _deranged_indices(length: int, seed: int) -> list[int]:
    if length < 2:
        raise ValueError("matched-unpaired control requires at least two pairs")
    rng = random.Random(seed)
    offset = rng.randrange(1, length)
    return [(index + offset) % length for index in range(length)]


def _encode_blocks(
    tokenizer: BatchTokenizer,
    texts: list[str],
    eos_id: int,
    vocab_size: int,
) -> list[np.ndarray]:
    encodings = tokenizer.encode_batch(texts, add_special_tokens=False)
    blocks = []
    for encoding in encodings:
        values = list(encoding.ids)
        values.append(eos_id)
        if not values:
            raise ValueError("tokenizer returned an empty block")
        if min(values) < 0 or max(values) >= vocab_size:
            raise ValueError("token ID outside declared vocabulary")
        blocks.append(np.asarray(values, dtype="<u2"))
    return blocks


def pack_intact_blocks(
    blocks: Iterable[np.ndarray],
    *,
    sequence_length: int,
    pad_id: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Pack without splitting any causal pair across training rows."""

    block_list = list(blocks)
    if not block_list:
        return (
            np.empty((0, sequence_length), dtype="<u2"),
            np.empty((0,), dtype="<u2"),
        )
    too_long = [len(block) for block in block_list if len(block) > sequence_length]
    if too_long:
        raise ValueError(
            f"{len(too_long)} pair blocks exceed sequence_length={sequence_length}; "
            f"maximum={max(too_long)}"
        )

    ordered = sorted(block_list, key=len)
    low = 0
    high = len(ordered) - 1
    packed: list[np.ndarray] = []
    lengths: list[int] = []
    while low <= high:
        pieces = [ordered[high]]
        used = len(ordered[high])
        high -= 1
        while low <= high and used + len(ordered[low]) <= sequence_length:
            pieces.append(ordered[low])
            used += len(ordered[low])
            low += 1
        row = np.full(sequence_length, pad_id, dtype="<u2")
        cursor = 0
        for piece in pieces:
            row[cursor:cursor + len(piece)] = piece
            cursor += len(piece)
        packed.append(row)
        lengths.append(used)
    return np.stack(packed), np.asarray(lengths, dtype="<u2")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _stats(
    token_path: Path,
    length_path: Path,
    *,
    sequence_length: int,
    block_lengths: list[int],
) -> PackedCorpusStats:
    lengths = np.memmap(length_path, dtype="<u2", mode="r")
    rows = len(lengths)
    valid_tokens = int(np.asarray(lengths, dtype=np.int64).sum())
    predicted_tokens = int(np.maximum(np.asarray(lengths, dtype=np.int64) - 1, 0).sum())
    total_slots = rows * sequence_length
    return PackedCorpusStats(
        rows=rows,
        valid_tokens=valid_tokens,
        predicted_tokens=predicted_tokens,
        padded_tokens=total_slots - valid_tokens,
        utilization=valid_tokens / total_slots if total_slots else 0.0,
        maximum_block_tokens=max(block_lengths) if block_lengths else 0,
        mean_block_tokens=float(np.mean(block_lengths)) if block_lengths else 0.0,
        sha256_tokens=_sha256(token_path),
        sha256_lengths=_sha256(length_path),
    )


def build_packed_corpora(
    *,
    tokenizer: BatchTokenizer,
    output_dir: Path,
    target_predicted_tokens: int,
    sequence_length: int,
    pad_id: int,
    eos_id: int,
    vocab_size: int,
    seed: int = 3407,
    chunk_pairs: int = 2_048,
    minimum_utilization: float = 0.80,
) -> dict[str, object]:
    if target_predicted_tokens <= 0:
        raise ValueError("target_predicted_tokens must be positive")
    if not 0.0 < minimum_utilization <= 1.0:
        raise ValueError("minimum_utilization must be in (0, 1]")
    if vocab_size > np.iinfo(np.uint16).max:
        raise ValueError("WorldPairs uint16 format requires vocab_size <= 65535")
    if not 0 <= pad_id < vocab_size or not 0 <= eos_id < vocab_size:
        raise ValueError("pad/eos token IDs must be inside the vocabulary")

    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "paired_tokens": output_dir / "paired.tokens.bin",
        "paired_lengths": output_dir / "paired.lengths.bin",
        "control_tokens": output_dir / "matched_unpaired.tokens.bin",
        "control_lengths": output_dir / "matched_unpaired.lengths.bin",
    }
    existing = [path for path in paths.values() if path.exists()]
    if existing:
        raise FileExistsError(
            "refusing to overwrite an existing WorldPairs corpus: "
            + ", ".join(str(path) for path in existing)
        )

    predicted = {"paired": 0, "matched_unpaired": 0}
    block_lengths = {"paired": [], "matched_unpaired": []}
    pair_count = 0
    chunk_id = 0
    next_index = {kind: 0 for kind in TRAIN_CONTRASTS}

    with (
        paths["paired_tokens"].open("wb") as paired_tokens,
        paths["paired_lengths"].open("wb") as paired_lengths,
        paths["control_tokens"].open("wb") as control_tokens,
        paths["control_lengths"].open("wb") as control_lengths,
    ):
        while min(predicted.values()) < target_predicted_tokens:
            generated = _balanced_chunk(
                start_by_kind=next_index,
                pair_count=chunk_pairs,
                seed=seed,
            )
            rendered = []
            for item in generated:
                pair_render = render_pair(item)
                validate_pair(item, pair_render)
                rendered.append(pair_render)

            derangement = _deranged_indices(
                len(rendered), seed + 1_000_003 * chunk_id
            )
            paired_texts = [
                _pair_block(item.factual, item.counterfactual)
                for item in rendered
            ]
            control_texts = [
                _pair_block(rendered[i].factual, rendered[j].counterfactual)
                for i, j in enumerate(derangement)
            ]

            paired_blocks = _encode_blocks(
                tokenizer, paired_texts, eos_id, vocab_size
            )
            control_blocks = _encode_blocks(
                tokenizer, control_texts, eos_id, vocab_size
            )
            block_lengths["paired"].extend(map(len, paired_blocks))
            block_lengths["matched_unpaired"].extend(map(len, control_blocks))

            for name, blocks, token_handle, length_handle in (
                ("paired", paired_blocks, paired_tokens, paired_lengths),
                (
                    "matched_unpaired",
                    control_blocks,
                    control_tokens,
                    control_lengths,
                ),
            ):
                rows, lengths = pack_intact_blocks(
                    blocks,
                    sequence_length=sequence_length,
                    pad_id=pad_id,
                )
                rows.tofile(token_handle)
                lengths.tofile(length_handle)
                predicted[name] += int(
                    np.maximum(lengths.astype(np.int64) - 1, 0).sum()
                )

            pair_count += len(generated)
            chunk_id += 1
            print(
                f"pairs={pair_count:,} "
                f"paired_tokens={predicted['paired']:,} "
                f"control_tokens={predicted['matched_unpaired']:,}",
                flush=True,
            )

    paired_stats = _stats(
        paths["paired_tokens"],
        paths["paired_lengths"],
        sequence_length=sequence_length,
        block_lengths=block_lengths["paired"],
    )
    control_stats = _stats(
        paths["control_tokens"],
        paths["control_lengths"],
        sequence_length=sequence_length,
        block_lengths=block_lengths["matched_unpaired"],
    )
    for name, stats in (("paired", paired_stats), ("matched_unpaired", control_stats)):
        if stats.utilization < minimum_utilization:
            raise RuntimeError(
                f"{name} packing utilization {stats.utilization:.3f} is below "
                f"minimum {minimum_utilization:.3f}; choose a better sequence length"
            )

    manifest: dict[str, object] = {
        "format": "worldpairs-fixed-uint16-v1",
        "generator_version": GENERATOR_VERSION,
        "seed": seed,
        "sequence_length": sequence_length,
        "vocab_size": vocab_size,
        "pad_id": pad_id,
        "eos_id": eos_id,
        "pairs_generated": pair_count,
        "train_contrasts": [kind.value for kind in TRAIN_CONTRASTS],
        "event_order_held_out": True,
        "corpora": {
            "paired": {
                "tokens_file": paths["paired_tokens"].name,
                "lengths_file": paths["paired_lengths"].name,
                **asdict(paired_stats),
            },
            "matched_unpaired": {
                "tokens_file": paths["control_tokens"].name,
                "lengths_file": paths["control_lengths"].name,
                **asdict(control_stats),
            },
        },
    }
    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


class PackedRowStore:
    """Random-access numpy view used by the Colab training mixer."""

    def __init__(self, manifest_path: Path, corpus: str) -> None:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != "worldpairs-fixed-uint16-v1":
            raise ValueError("unsupported WorldPairs manifest")
        entry = manifest["corpora"][corpus]
        root = manifest_path.parent
        self.sequence_length = int(manifest["sequence_length"])
        self.pad_id = int(manifest["pad_id"])
        self.rows = int(entry["rows"])
        self.tokens = np.memmap(
            root / entry["tokens_file"],
            dtype="<u2",
            mode="r",
            shape=(self.rows, self.sequence_length),
        )
        self.lengths = np.memmap(
            root / entry["lengths_file"], dtype="<u2", mode="r"
        )
        if len(self.lengths) != self.rows:
            raise ValueError("WorldPairs token/length row mismatch")

    @property
    def mean_predicted_tokens(self) -> float:
        return float(np.maximum(np.asarray(self.lengths, dtype=np.float64) - 1, 0).mean())

    def sample_numpy(
        self, rng: np.random.Generator, count: int
    ) -> tuple[np.ndarray, np.ndarray]:
        indices = rng.integers(0, self.rows, size=count)
        inputs = np.asarray(self.tokens[indices], dtype=np.int64)
        labels = inputs.copy()
        lengths = np.asarray(self.lengths[indices], dtype=np.int64)
        positions = np.arange(self.sequence_length)[None, :]
        labels[positions >= lengths[:, None]] = -100
        return inputs, labels


## 9. Generate fixed ID and OOD evaluations

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 9 — ID/OOD test generator
"""Leakage-resistant ID and OOD evaluation suite for WorldPairs."""

from __future__ import annotations

import hashlib
import json
from pathlib import Path



EVAL_GENERATOR_VERSION = "worldpairs-eval-v1.0"


def _answer_text(example: WorldExample, answer_id: str) -> str:
    if answer_id in {"open", "closed", "on", "off", "none", "nobody"}:
        return answer_id
    return example.initial_state.entity_by_id[answer_id].name


def evaluation_record(generated: GeneratedPair) -> dict[str, object]:
    rendered = render_pair(generated)
    validate_pair(generated, rendered)
    example = generated.pair.factual
    candidate_texts = [
        _answer_text(example, answer_id)
        for answer_id in generated.candidate_answer_ids
    ]
    return {
        "family_id": example.pair_family_id,
        "split": generated.split,
        "contrast_kind": generated.pair.intervention.kind.value,
        "reasoning_profile": generated.reasoning_profile,
        "chain_depth": generated.chain_depth,
        "candidate_answer_ids": list(generated.candidate_answer_ids),
        "candidate_texts": candidate_texts,
        "prompt_a": rendered.factual.prompt,
        "prompt_b": rendered.counterfactual.prompt,
        "answer_id_a": generated.pair.factual.answer_id,
        "answer_id_b": generated.pair.counterfactual.answer_id,
        "answer_text_a": rendered.factual.answer_text,
        "answer_text_b": rendered.counterfactual.answer_text,
        "intervention": {
            "kind": generated.pair.intervention.kind.value,
            "event_index": generated.pair.intervention.event_index,
            "field_name": generated.pair.intervention.field_name,
            "old_value": generated.pair.intervention.old_value,
            "new_value": generated.pair.intervention.new_value,
        },
    }


def build_evaluation_suite(
    *,
    output_path: Path,
    pairs_per_cell: int,
    seed: int = 93_411,
) -> dict[str, object]:
    if pairs_per_cell <= 0:
        raise ValueError("pairs_per_cell must be positive")
    if output_path.exists():
        raise FileExistsError(f"refusing to overwrite evaluation suite: {output_path}")

    groups: dict[str, list[GeneratedPair]] = {
        "id": [],
        "ood_template": [],
        "ood_depth": [],
        "ood_kind": [],
    }
    train_kinds = (
        ContrastKind.DESTINATION,
        ContrastKind.HOLDER,
        ContrastKind.CONTAINER,
        ContrastKind.PROPERTY_STATE,
    )
    for split in ("id", "ood_template"):
        for kind in train_kinds:
            for index in range(pairs_per_cell):
                groups[split].append(
                    generate_pair(
                        index=index,
                        kind=kind,
                        split=split,
                        seed=seed,
                    )
                )
    for kind in (ContrastKind.DESTINATION, ContrastKind.HOLDER):
        for index in range(pairs_per_cell):
            groups["ood_depth"].append(
                generate_pair(
                    index=index,
                    kind=kind,
                    split="ood_depth",
                    seed=seed,
                )
            )
    for index in range(pairs_per_cell * 2):
        groups["ood_kind"].append(
            generate_pair(
                index=index,
                kind=ContrastKind.EVENT_ORDER,
                split="ood_kind",
                seed=seed,
            )
        )

    assert_disjoint_families(groups)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    digest = hashlib.sha256()
    with output_path.open("x", encoding="utf-8") as handle:
        for split in ("id", "ood_template", "ood_depth", "ood_kind"):
            for generated in groups[split]:
                record = evaluation_record(generated)
                line = json.dumps(record, ensure_ascii=False, sort_keys=True)
                handle.write(line + "\n")
                digest.update((line + "\n").encode("utf-8"))
                count += 1

    manifest = {
        "format": "worldpairs-evaluation-jsonl-v1",
        "generator_version": EVAL_GENERATOR_VERSION,
        "seed": seed,
        "pairs_per_cell": pairs_per_cell,
        "records": count,
        "sha256": digest.hexdigest(),
        "splits": {name: len(items) for name, items in groups.items()},
        "path": output_path.name,
    }
    manifest_path = output_path.with_suffix(".manifest.json")
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def load_evaluation_suite(path: Path) -> list[dict[str, object]]:
    records = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]
    family_ids = [record["family_id"] for record in records]
    if len(family_ids) != len(set(family_ids)):
        raise ValueError("duplicate family_id in evaluation suite")
    return records


## 10. Run generator-consistency preflight checks

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 10 — Quick preflight: is the generator internally consistent?
def run_preflight(samples_per_case: int = 50) -> None:
    train_kinds = (
        ContrastKind.DESTINATION,
        ContrastKind.HOLDER,
        ContrastKind.CONTAINER,
        ContrastKind.PROPERTY_STATE,
    )
    cases = [(kind, split) for split in ("train", "id", "ood_template") for kind in train_kinds]
    cases += [
        (ContrastKind.DESTINATION, "ood_depth"),
        (ContrastKind.HOLDER, "ood_depth"),
        (ContrastKind.EVENT_ORDER, "ood_kind"),
    ]
    family_ids = set()
    started = time.perf_counter()
    for kind, split in cases:
        for index in range(samples_per_case):
            generated = generate_pair(index=index, kind=kind, split=split, seed=SEED)
            rendered = render_pair(generated)
            report = validate_pair(generated, rendered)
            assert report.family_id not in family_ids, f"Family leakage: {report.family_id}"
            family_ids.add(report.family_id)

    probe = generate_pair(index=999_999, kind=ContrastKind.HOLDER, split="train", seed=SEED)
    wrong = replace(probe.pair.factual, answer_id=probe.pair.counterfactual.answer_id)
    tampered = replace(probe, pair=replace(probe.pair, factual=wrong))
    try:
        validate_pair(tampered)
        raise AssertionError("Invalid example was not detected")
    except PairValidationError:
        pass

    elapsed = time.perf_counter() - started
    print(f"PASS — {len(family_ids):,} pairs {elapsed:.2f} verified in seconds.")


run_preflight()
example = generate_pair(index=0, kind=ContrastKind.DESTINATION, split="ood_depth", seed=SEED)
print("\nExample verified pair:\n")
print(render_pair(example).block_text)


## 11. Verify tokenizer coverage and pair lengths

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 11 — Tokenizer and pair-length check
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
VOCAB_SIZE = tokenizer.get_vocab_size()
PAD_ID = tokenizer.token_to_id("<pad>")
EOS_ID = tokenizer.token_to_id("<eos>")

assert VOCAB_SIZE == 15_360, f"Expected vocab=15360, found={VOCAB_SIZE}"
assert PAD_ID is not None and EOS_ID is not None

texts = []
kinds = (
    ContrastKind.DESTINATION,
    ContrastKind.HOLDER,
    ContrastKind.CONTAINER,
    ContrastKind.PROPERTY_STATE,
)
for index in range(250):
    generated = generate_pair(
        index=index // len(kinds),
        kind=kinds[index % len(kinds)],
        split="train",
        seed=SEED,
    )
    rendered = render_pair(generated)
    texts.append(_pair_block(rendered.factual, rendered.counterfactual))

encoded = tokenizer.encode_batch(texts, add_special_tokens=False)
lengths = np.asarray([len(item.ids) + 1 for item in encoded])
assert lengths.max() <= SEQ_LEN, (
    f"Pair {lengths.max()} token; SEQ_LEN={SEQ_LEN}. Pair cannot be split."
)
print("PASS — vocab:", VOCAB_SIZE, "| most long pair:", int(lengths.max()), "token")
del texts, encoded, lengths
gc.collect()


## 12. Build the fixed ID/OOD test set

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 12 — Build one fixed ID/OOD test set
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_evaluation_suite(path: Path, pairs_per_cell: int, seed: int) -> Path:
    manifest_path = path.with_suffix(".manifest.json")
    if path.exists() or manifest_path.exists():
        assert path.exists() and manifest_path.exists(), "Test set is incomplete"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        assert manifest["format"] == "worldpairs-evaluation-jsonl-v1"
        assert int(manifest["seed"]) == seed
        assert int(manifest["pairs_per_cell"]) == pairs_per_cell
        assert sha256_file(path) == manifest["sha256"], "Test-set hash mismatch"
        print(f"Existing test set verified: {manifest['records']:,} pairs")
        return path

    manifest = build_evaluation_suite(
        output_path=path,
        pairs_per_cell=pairs_per_cell,
        seed=seed,
    )
    print(f"Test set created: {manifest['records']:,} pairs")
    return path


PROOF_SUITE = ensure_evaluation_suite(
    WORLD_ROOT / "evaluation/proof.jsonl",
    pairs_per_cell=100,
    seed=EVAL_SEED,
)


## 13. Prepare matched W3 and wrong-pair contents

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.


In [ ]:
# 13 — W3 and D1 for prepare the same synthetic contents first

MIN_PACKING_UTILIZATION = 0.65


def verify_packed_manifest(
    manifest_path: Path,
    required_tokens: int,
) -> dict:
    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    assert manifest["format"] == "worldpairs-fixed-uint16-v1"
    assert int(manifest["sequence_length"]) == SEQ_LEN
    assert int(manifest["vocab_size"]) == VOCAB_SIZE

    for entry in manifest["corpora"].values():
        token_path = (
            manifest_path.parent / entry["tokens_file"]
        )
        length_path = (
            manifest_path.parent / entry["lengths_file"]
        )

        assert token_path.exists()
        assert length_path.exists()
        assert (
            sha256_file(token_path)
            == entry["sha256_tokens"]
        )
        assert (
            sha256_file(length_path)
            == entry["sha256_lengths"]
        )
        assert (
            int(entry["predicted_tokens"])
            >= required_tokens
        )
        assert (
            float(entry["utilization"])
            >= MIN_PACKING_UTILIZATION
        )

    return manifest


def recover_completed_binary_corpus(
    folder: Path,
    manifest_path: Path,
    required_tokens: int,
) -> bool:
    files = {
        "paired": (
            folder / "paired.tokens.bin",
            folder / "paired.lengths.bin",
        ),
        "matched_unpaired": (
            folder / "matched_unpaired.tokens.bin",
            folder / "matched_unpaired.lengths.bin",
        ),
    }

    existing = [
        path
        for pair in files.values()
        for path in pair
        if path.exists()
    ]

    if not existing:
        return False

    complete = all(
        path.exists()
        for pair in files.values()
        for path in pair
    )

    entries = {}

    if complete:
        for name, (
            token_path,
            length_path,
        ) in files.items():
            item_size = np.dtype("<u2").itemsize

            if length_path.stat().st_size % item_size != 0:
                complete = False
                break

            lengths = np.fromfile(
                length_path,
                dtype="<u2",
            ).astype(np.int64)

            rows = int(len(lengths))
            expected_bytes = (
                rows
                * SEQ_LEN
                * item_size
            )

            if (
                rows == 0
                or token_path.stat().st_size
                != expected_bytes
                or np.any(lengths <= 1)
                or np.any(lengths > SEQ_LEN)
            ):
                complete = False
                break

            valid_tokens = int(lengths.sum())
            predicted_tokens = int(
                (lengths - 1).sum()
            )

            entries[name] = {
                "tokens_file": token_path.name,
                "lengths_file": length_path.name,
                "rows": rows,
                "valid_tokens": valid_tokens,
                "predicted_tokens": predicted_tokens,
                "padded_tokens": int(
                    rows * SEQ_LEN - valid_tokens
                ),
                "utilization": float(
                    valid_tokens
                    / (rows * SEQ_LEN)
                ),
                "maximum_block_tokens": None,
                "mean_block_tokens": None,
                "sha256_tokens": sha256_file(
                    token_path
                ),
                "sha256_lengths": sha256_file(
                    length_path
                ),
            }

        if complete:
            complete = all(
                entry["predicted_tokens"]
                >= required_tokens
                and entry["utilization"]
                >= MIN_PACKING_UTILIZATION
                for entry in entries.values()
            )

    if complete:
        recovered_manifest = {
            "format": "worldpairs-fixed-uint16-v1",
            "generator_version": GENERATOR_VERSION,
            "seed": SEED,
            "sequence_length": SEQ_LEN,
            "vocab_size": VOCAB_SIZE,
            "pad_id": PAD_ID,
            "eos_id": EOS_ID,
            "pairs_generated": None,
            "train_contrasts": [
                kind.value
                for kind in TRAIN_CONTRASTS
            ],
            "event_order_held_out": True,
            "recovered_after_utilization_guard": True,
            "corpora": entries,
        }

        temporary_path = (
            manifest_path.with_suffix(".tmp")
        )

        temporary_path.write_text(
            json.dumps(
                recovered_manifest,
                indent=2,
            ),
            encoding="utf-8",
        )

        temporary_path.replace(manifest_path)

        print(
            "existing binary corpus "
            "was recovered without deletion."
        )

        return True

    print(
        "Incomplete binary corpus found; "
        "rebuilding missing files."
    )

    for path in existing:
        path.unlink()

    if manifest_path.exists():
        manifest_path.unlink()

    return False


REQUIRED_WORLD_TOKENS = math.ceil(
    TOKEN_BUDGET
    * WORLD_RATIO
    * 1.15
)

CORPUS_DIR = (
    WORLD_ROOT
    / (
        f"corpus_"
        f"{math.ceil(REQUIRED_WORLD_TOKENS / 1_000_000)}m"
        f"_seq{SEQ_LEN}"
    )
)

WORLD_MANIFEST = (
    CORPUS_DIR / "manifest.json"
)


if WORLD_MANIFEST.exists():
    packed_manifest = verify_packed_manifest(
        WORLD_MANIFEST,
        REQUIRED_WORLD_TOKENS,
    )

    print(
        "Existing WorldPairs corpus verified."
    )

else:
    recovered = recover_completed_binary_corpus(
        CORPUS_DIR,
        WORLD_MANIFEST,
        REQUIRED_WORLD_TOKENS,
    )

    if not recovered:
        print(
            f"{REQUIRED_WORLD_TOKENS:,} "
            "preparing synthetic tokens..."
        )

        build_packed_corpora(
            tokenizer=tokenizer,
            output_dir=CORPUS_DIR,
            target_predicted_tokens=(
                REQUIRED_WORLD_TOKENS
            ),
            sequence_length=SEQ_LEN,
            pad_id=PAD_ID,
            eos_id=EOS_ID,
            vocab_size=VOCAB_SIZE,
            seed=SEED,
            chunk_pairs=2_048,
            minimum_utilization=(
                MIN_PACKING_UTILIZATION
            ),
        )

    packed_manifest = verify_packed_manifest(
        WORLD_MANIFEST,
        REQUIRED_WORLD_TOKENS,
    )


for name, entry in packed_manifest[
    "corpora"
].items():
    print(
        name,
        "|",
        f"{entry['predicted_tokens']:,} token",
        "|",
        f"doluluk={entry['utilization']:.1%}",
    )

## 14. Define the shared C2 model

Defines or validates model components used by the experiment. Keep tensor shapes, parameter counts, and causal behavior unchanged when reproducing the run.


In [ ]:
# 14 — C2 model
@dataclass
class C2Config:
    name: str = "W3"
    vocab_size: int = 15_360
    max_seq_len: int = 2_048
    d_model: int = 448
    n_layers: int = 20
    n_heads: int = 7
    n_kv_heads: int = 1
    d_ff: int = 1_216
    rope_theta: float = 10_000.0
    norm_eps: float = 1e-5
    dropout: float = 0.0


@dataclass
class LMOutput:
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None


class C2RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        y = x.float()
        y = y * torch.rsqrt(y.square().mean(dim=-1, keepdim=True) + self.eps)
        return y.to(dtype) * self.weight.to(dtype=dtype)


class C2RoPE(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, theta: float):
        super().__init__()
        assert head_dim % 2 == 0
        inv = 1.0 / (
            theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        sequence = x.shape[-2]
        cos = self.cos[:sequence].to(device=x.device, dtype=x.dtype)[None, None]
        sin = self.sin[:sequence].to(device=x.device, dtype=x.dtype)[None, None]
        even, odd = x[..., 0::2], x[..., 1::2]
        result = torch.empty_like(x)
        result[..., 0::2] = even * cos - odd * sin
        result[..., 1::2] = even * sin + odd * cos
        return result


class C2Attention(nn.Module):
    def __init__(self, cfg: C2Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.dropout = cfg.dropout
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.rope = C2RoPE(self.head_dim, cfg.max_seq_len, cfg.rope_theta)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, sequence, _ = x.shape
        q = self.q_proj(x).view(
            batch, sequence, self.n_heads, self.head_dim
        ).transpose(1, 2)
        k = self.k_proj(x).view(
            batch, sequence, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)
        v = self.v_proj(x).view(
            batch, sequence, self.n_kv_heads, self.head_dim
        ).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        y = F.scaled_dot_product_attention(
            q, k, v,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=True,
            enable_gqa=(self.n_heads != self.n_kv_heads),
        )
        y = y.transpose(1, 2).contiguous().view(batch, sequence, -1)
        return self.o_proj(y)


class C2SwiGLU(nn.Module):
    def __init__(self, cfg: C2Config):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))


class C2Block(nn.Module):
    def __init__(self, cfg: C2Config):
        super().__init__()
        self.attn_norm = C2RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = C2Attention(cfg)
        self.ffn_norm = C2RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = C2SwiGLU(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x))
        x = x + self.ffn(self.ffn_norm(x))
        return x


class C2WorldPairsLM(nn.Module):
    PARAMETER_COUNT = 48_760_768

    def __init__(self, cfg: Optional[C2Config] = None):
        super().__init__()
        self.config = cfg or C2Config()
        self.token_embedding = nn.Embedding(
            self.config.vocab_size, self.config.d_model
        )
        self.layers = nn.ModuleList(
            C2Block(self.config) for _ in range(self.config.n_layers)
        )
        self.final_norm = C2RMSNorm(
            self.config.d_model, self.config.norm_eps
        )
        self.apply(self._initialize)
        actual = self.num_trainable_parameters()
        assert actual == self.PARAMETER_COUNT, (
            f"C2 parameter mismatch: {actual:,} != {self.PARAMETER_COUNT:,}"
        )

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, C2RMSNorm):
            nn.init.ones_(module.weight)

    def num_trainable_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
    ) -> LMOutput:
        if input_ids.ndim != 2 or input_ids.dtype != torch.long:
            raise TypeError("input_ids [B,S] must be torch.long")
        if input_ids.shape[1] > self.config.max_seq_len:
            raise ValueError("Sequence exceeds the RoPE cache limit")
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        logits = F.linear(x, self.token_embedding.weight)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].contiguous().view(-1, self.config.vocab_size),
                labels[:, 1:].contiguous().view(-1),
                ignore_index=-100,
            )
        return LMOutput(logits=logits, loss=loss)


def build_c2_worldpairs() -> C2WorldPairsLM:
    return C2WorldPairsLM(C2Config(vocab_size=VOCAB_SIZE, max_seq_len=2_048))


## 15. Run GPU structural tests

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 15 — GPU structural test
def clear_cuda() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


clear_cuda()
seed_everything(SEED)
probe_model = build_c2_worldpairs().to(DEVICE)
assert probe_model.num_trainable_parameters() == 48_760_768
assert not hasattr(probe_model, "lm_head"), "A separate lm_head breaks weight tying"

left = torch.randint(0, VOCAB_SIZE, (2, 32), device=DEVICE)
right = left.clone()
right[:, 17:] = torch.randint(0, VOCAB_SIZE, right[:, 17:].shape, device=DEVICE)
probe_model.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    logits_left = probe_model(left).logits[:, :17].float()
    logits_right = probe_model(right).logits[:, :17].float()
causal_delta = float((logits_left - logits_right).abs().max())
assert causal_delta < 0.02, f"Causal leakage: {causal_delta}"

probe_model.train()
labels = left.clone()
with torch.autocast("cuda", dtype=torch.bfloat16):
    output = probe_model(left, labels=labels)
assert output.logits.shape == (2, 32, VOCAB_SIZE)
assert torch.isfinite(output.loss)
output.loss.backward()
grad_norm = float(torch.nn.utils.clip_grad_norm_(probe_model.parameters(), 1.0))
assert math.isfinite(grad_norm) and grad_norm > 0

print("parameter       :", f"{probe_model.num_trainable_parameters():,}")
print("Logits          :", tuple(output.logits.shape))
print("Loss            :", float(output.loss))
print("Gradient norm   :", grad_norm)
print("Causal fark     :", causal_delta)
print("Structural result   : PASS")
del probe_model, left, right, labels, output, logits_left, logits_right
clear_cuda()


## 16. Build the fast natural-shard loader

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 16 — Fast natural-shard loader

def resolve_path(
    root: Path,
    value: str,
) -> Optional[Path]:
    raw = Path(value)

    for candidate in (
        raw,
        root / raw,
        root / raw.name,
    ):
        if candidate.exists():
            return candidate.resolve()

    return None


def resolve_natural_splits(
    manifest_path: Path,
) -> tuple[list[Path], list[Path]]:
    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    train_paths = []
    validation_paths = []

    if isinstance(manifest.get("shards"), list):
        for item in manifest["shards"]:
            if not isinstance(item, dict):
                continue

            value = (
                item.get("file")
                or item.get("path")
            )

            if not isinstance(value, str):
                continue

            path = resolve_path(
                manifest_path.parent,
                value,
            )

            if path is None:
                continue

            split = str(
                item.get("split", "train")
            ).lower()

            if split in {
                "validation",
                "val",
                "dev",
            }:
                validation_paths.append(path)
            else:
                train_paths.append(path)

    if not train_paths or not validation_paths:
        candidates = (
            sorted(
                manifest_path.parent.glob("*.bin")
            )
            + sorted(
                manifest_path.parent.glob("*.npy")
            )
        )

        validation_named = [
            path
            for path in candidates
            if any(
                tag in path.name.lower()
                for tag in (
                    "validation",
                    "valid",
                    "val",
                )
            )
        ]

        train_named = [
            path
            for path in candidates
            if path not in validation_named
        ]

        if train_named and validation_named:
            train_paths = train_named
            validation_paths = validation_named

    if not train_paths or not validation_paths:
        raise RuntimeError(
            "Train/validation shard split not found. "
            f"Manifest: {manifest_path}"
        )

    return train_paths, validation_paths


def open_token_array(path: Path):
    if path.suffix == ".npy":
        return np.load(
            path,
            mmap_mode="r",
        )

    return np.memmap(
        path,
        dtype="<u2",
        mode="r",
    )


TRAIN_SHARDS, VALIDATION_SHARDS = (
    resolve_natural_splits(
        NATURAL_MANIFEST
    )
)

print(
    "Train shard      :",
    len(TRAIN_SHARDS),
)

print(
    "Validation shard :",
    len(VALIDATION_SHARDS),
)

print(
    "data konumu      :",
    (
        "Google Drive"
        if str(TRAIN_SHARDS[0]).startswith(
            "/content/drive"
        )
        else "Yerel disk"
    ),
)


class NaturalRowStore:
    def __init__(
        self,
        paths: list[Path],
        sequence_length: int,
    ):
        self.sequence_length = sequence_length

        self.arrays = [
            open_token_array(path)
            for path in paths
        ]

        self.lengths = np.asarray(
            [
                len(array)
                for array in self.arrays
            ],
            dtype=np.int64,
        )

        if np.all(
            self.lengths < sequence_length
        ):
            raise ValueError(
                "No natural shard large enough for the sequence"
            )

    def sample_numpy(
        self,
        rng: np.random.Generator,
        count: int,
    ) -> tuple[np.ndarray, np.ndarray]:
        required_tokens = (
            count
            * self.sequence_length
        )

        usable = np.maximum(
            self.lengths
            - required_tokens
            + 1,
            0,
        ).astype(np.float64)

        if usable.sum() <= 0:
            raise ValueError(
                "No shard large enough for one batch"
            )

        probabilities = (
            usable
            / usable.sum()
        )

        shard_id = int(
            rng.choice(
                len(self.arrays),
                p=probabilities,
            )
        )

        array = self.arrays[shard_id]

        maximum_start = (
            len(array)
            - required_tokens
        )

        start = int(
            rng.integers(
                0,
                maximum_start + 1,
            )
        )

        block = np.asarray(
            array[
                start:
                start + required_tokens
            ],
            dtype=np.int64,
        ).copy()

        rows = block.reshape(
            count,
            self.sequence_length,
        )

        return rows, rows.copy()


NATURAL_TRAIN = NaturalRowStore(
    TRAIN_SHARDS,
    SEQ_LEN,
)


def fixed_natural_validation(
    batch_size: int = 8,
    batches: int = 40,
) -> list[torch.Tensor]:
    arrays = [
        open_token_array(path)
        for path in VALIDATION_SHARDS
    ]

    required_tokens = (
        batch_size
        * SEQ_LEN
    )

    arrays = [
        array
        for array in arrays
        if len(array) >= required_tokens
    ]

    if not arrays:
        raise RuntimeError(
            "No shard large enough for validation"
        )

    result = []

    for batch_index in range(batches):
        array = arrays[
            batch_index
            % len(arrays)
        ]

        maximum_start = (
            len(array)
            - required_tokens
        )

        fraction = (
            batch_index + 0.5
        ) / batches

        start = int(
            fraction
            * maximum_start
        )

        block = np.asarray(
            array[
                start:
                start + required_tokens
            ],
            dtype=np.int64,
        ).copy()

        rows = block.reshape(
            batch_size,
            SEQ_LEN,
        )

        result.append(
            torch.from_numpy(rows)
        )

    return result


NATURAL_VALIDATION = (
    fixed_natural_validation()
)


loader_test_start = time.perf_counter()

test_inputs, test_labels = (
    NATURAL_TRAIN.sample_numpy(
        np.random.default_rng(SEED),
        MICRO_BATCH,
    )
)

loader_test_seconds = (
    time.perf_counter()
    - loader_test_start
)

assert test_inputs.shape == (
    MICRO_BATCH,
    SEQ_LEN,
)

assert test_labels.shape == (
    MICRO_BATCH,
    SEQ_LEN,
)

print(
    "fixed validation :",
    len(NATURAL_VALIDATION),
    "batch",
)

print(
    "Loader testi     :",
    f"{loader_test_seconds:.3f} saniye/batch",
)

print(
    "Loader result    : PASS"
)

del test_inputs
del test_labels
gc.collect()

## 17. Define the matched D0/D1/W3 batch mixer

Prepares or validates persistent data artifacts. Review the Drive paths before running; large corpus operations can take time and may replace files at the configured destination.


In [ ]:
# 17 — D0/D1/W3 adil batch mixer
class MixedBatchSource:
    def __init__(
        self,
        *,
        natural: NaturalRowStore,
        world: Optional[PackedRowStore],
        desired_world_token_ratio: float,
        batch_size: int,
        seed: int,
    ):
        if not 0.0 <= desired_world_token_ratio < 1.0:
            raise ValueError("world token ratio must be in [0,1)")
        if desired_world_token_ratio > 0 and world is None:
            raise ValueError("A WorldPairs store is required for a positive ratio")
        self.natural = natural
        self.world = world
        self.ratio = desired_world_token_ratio
        self.batch_size = batch_size
        # Separate RNG streams ensure D0/D1/W3 see the same natural rows;
        # synthetic sampling does not shift natural-data order.
        self.natural_rng = np.random.default_rng(seed)
        self.world_rng = np.random.default_rng(seed + 1_000_003)
        self.position_rng = np.random.default_rng(seed + 2_000_033)
        self.carry = 0.0

        natural_tokens = natural.sequence_length - 1
        if world is None or desired_world_token_ratio == 0:
            self.world_row_probability = 0.0
        else:
            world_tokens = world.mean_predicted_tokens
            self.world_row_probability = (
                desired_world_token_ratio * natural_tokens / world_tokens
            )
            if self.world_row_probability > 1.0:
                raise ValueError(
                    "Requested synthetic-token ratio this pair-row doluluguyla "
                    "cannot be achieved. Reduce the sequence length."
                )

    def state_dict(self) -> dict:
        return {
            "natural_rng_state": self.natural_rng.bit_generator.state,
            "world_rng_state": self.world_rng.bit_generator.state,
            "position_rng_state": self.position_rng.bit_generator.state,
            "carry": self.carry,
        }

    def load_state_dict(self, state: dict) -> None:
        self.natural_rng.bit_generator.state = state["natural_rng_state"]
        self.world_rng.bit_generator.state = state["world_rng_state"]
        self.position_rng.bit_generator.state = state["position_rng_state"]
        self.carry = float(state["carry"])

    def next(self) -> tuple[torch.Tensor, torch.Tensor, dict[str, int]]:
        target = self.batch_size * self.world_row_probability + self.carry
        world_rows = int(math.floor(target))
        self.carry = target - world_rows
        world_rows = min(max(world_rows, 0), self.batch_size)
        # Each row is first filled with full-length natural tokens. WorldPairs
        # satirlarinda only the leading, intact pair blocks bunun
        # are overwritten. The EOS after a pair separates the natural tail. Therefore D0,
        # D1 and W3 see exactly the same number of target tokens at every optimizer step;
        # padding, more small efektif batch or there is no extra-update confound.
        inputs, _ = self.natural.sample_numpy(
            self.natural_rng, self.batch_size
        )
        world_predicted = 0
        if world_rows:
            assert self.world is not None
            world_inputs, world_labels = self.world.sample_numpy(
                self.world_rng, world_rows
            )
            positions = self.position_rng.choice(
                self.batch_size, size=world_rows, replace=False
            )
            lengths = (world_labels != -100).sum(axis=1).astype(np.int64)
            for source_row, destination_row, length in zip(
                range(world_rows), positions, lengths
            ):
                inputs[destination_row, :length] = world_inputs[
                    source_row, :length
                ]
            world_predicted = int(np.maximum(lengths - 1, 0).sum())

        labels = inputs.copy()
        metrics = {
            "predicted_tokens": self.batch_size * (SEQ_LEN - 1),
            "world_predicted_tokens": int(world_predicted),
            "world_rows": world_rows,
        }
        return torch.from_numpy(inputs), torch.from_numpy(labels), metrics


## 18. Define training and checkpoint-resume logic

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.

> **Destructive path operation:** this cell removes or replaces files at an explicit path. Verify the destination before execution.


In [ ]:
# 18 — Training and checkpointing
PEAK_LR = 2.5e-3
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
LOG_EVERY = 10
SAVE_EVERY = 250
VALIDATE_EVERY = 500


@dataclass(frozen=True)
class RunSpec:
    stage: str
    arm: str
    token_budget: int
    seed: int
    world_ratio: float
    world_corpus: Optional[str]
    world_manifest: Optional[str]

    @property
    def run_dir(self) -> Path:
        return WORLD_ROOT / "runs" / self.stage / self.arm / f"seed_{self.seed}"


def lr_at_tokens(seen_tokens: int, token_budget: int) -> float:
    warmup = max(1, int(token_budget * WARMUP_RATIO))
    if seen_tokens < warmup:
        return PEAK_LR * (seen_tokens + 1) / warmup
    progress = (seen_tokens - warmup) / max(1, token_budget - warmup)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return PEAK_LR * (
        MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine
    )


@torch.inference_mode()
def evaluate_natural_loss(model: nn.Module) -> dict[str, float]:
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    started = time.perf_counter()
    for cpu_tokens in NATURAL_VALIDATION:
        tokens = cpu_tokens.to(DEVICE, dtype=torch.long, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output = model(tokens, labels=tokens)
        count = tokens.shape[0] * (tokens.shape[1] - 1)
        total_nll += float(output.loss) * count
        total_tokens += count
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    model.train()
    loss = total_nll / total_tokens
    return {
        "loss": loss,
        "perplexity": math.exp(min(loss, 20.0)),
        "tokens": total_tokens,
        "tokens_per_second": total_tokens / elapsed,
    }


def save_training_checkpoint(
    path: Path,
    *,
    raw_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    spec: RunSpec,
    step: int,
    seen_tokens: int,
    seen_world_tokens: int,
    history: list[dict],
    source: MixedBatchSource,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(
        {
            "format": "anka-w3-checkpoint-v1",
            "spec": asdict(spec),
            "model_config": asdict(raw_model.config),
            "model": raw_model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "step": step,
            "seen_tokens": seen_tokens,
            "seen_world_tokens": seen_world_tokens,
            "history": history,
            "source_state": source.state_dict(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all(),
            "python_rng_state": random.getstate(),
        },
        temporary,
    )
    temporary.replace(path)


def save_final_checkpoint(
    path: Path,
    *,
    raw_model: nn.Module,
    spec: RunSpec,
    step: int,
    seen_tokens: int,
    seen_world_tokens: int,
    history: list[dict],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(
        {
            "format": "anka-w3-final-v1",
            "spec": asdict(spec),
            "model_config": asdict(raw_model.config),
            "model": raw_model.state_dict(),
            "step": step,
            "seen_tokens": seen_tokens,
            "seen_world_tokens": seen_world_tokens,
            "history": history,
        },
        temporary,
    )
    temporary.replace(path)


def train_run(spec: RunSpec) -> Path:
    run_dir = spec.run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    latest_path = run_dir / "latest.pt"
    final_path = run_dir / "final.pt"
    result_path = run_dir / "training_result.json"
    if final_path.exists() and result_path.exists():
        print(f"{spec.arm}: completed checkpoint found, skipped.")
        return final_path

    clear_cuda()
    seed_everything(spec.seed)
    world_store = None
    if spec.world_corpus is not None:
        if spec.world_manifest is None:
            raise ValueError("world_corpus requires a manifest")
        world_store = PackedRowStore(
            Path(spec.world_manifest), spec.world_corpus
        )
    source = MixedBatchSource(
        natural=NATURAL_TRAIN,
        world=world_store,
        desired_world_token_ratio=spec.world_ratio,
        batch_size=MICRO_BATCH,
        seed=spec.seed + 17,
    )

    raw_model = build_c2_worldpairs().to(DEVICE)
    optimizer = torch.optim.AdamW(
        raw_model.parameters(),
        lr=PEAK_LR,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=WEIGHT_DECAY,
        fused=True,
    )
    step = 0
    seen_tokens = 0
    seen_world_tokens = 0
    history: list[dict] = []

    if latest_path.exists() and not final_path.exists():
        checkpoint = torch.load(
            latest_path, map_location="cpu", weights_only=False
        )
        if checkpoint.get("format") != "anka-w3-checkpoint-v1":
            raise RuntimeError(f"earlier/uyumsuz checkpoint: {latest_path}")
        if checkpoint["spec"] != asdict(spec):
            raise RuntimeError(
                "Checkpoint experiment definition differs from this cell; "
                "use a new arm/stage name instead of overwriting."
            )
        raw_model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        step = int(checkpoint["step"])
        seen_tokens = int(checkpoint["seen_tokens"])
        seen_world_tokens = int(checkpoint["seen_world_tokens"])
        history = list(checkpoint.get("history", []))
        source.load_state_dict(checkpoint["source_state"])
        torch.set_rng_state(checkpoint["torch_rng_state"])
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])
        random.setstate(checkpoint["python_rng_state"])
        print(f"{spec.arm}: step={step:,}, token={seen_tokens:,} resuming from.")

    train_model = raw_model
    if USE_TORCH_COMPILE:
        print("torch.compile enabled; the first step may compile for several minutes.")
        train_model = torch.compile(raw_model, dynamic=False, mode="default")

    print("\n" + "=" * 88)
    print(f"{spec.stage} / {spec.arm}")
    print("=" * 88)
    print("parameter          :", f"{raw_model.num_trainable_parameters():,}")
    print("target token        :", f"{spec.token_budget:,}")
    print("World-token ratio  :", f"{spec.world_ratio:.1%}")
    print("World-row probability:", f"{source.world_row_probability:.4f}")
    print("Corpus             :", spec.world_corpus or "natural-only")

    raw_model.train()
    torch.cuda.reset_peak_memory_stats()
    interval_start = time.perf_counter()
    interval_effective_tokens = 0
    interval_positions = 0
    interval_world_tokens = 0
    interval_losses = []
    interval_data = 0.0
    interval_steps = 0
    started = time.time()

    while seen_tokens < spec.token_budget:
        lr = lr_at_tokens(seen_tokens, spec.token_budget)
        for group in optimizer.param_groups:
            group["lr"] = lr

        data_started = time.perf_counter()
        cpu_inputs, cpu_labels, batch_metrics = source.next()
        interval_data += time.perf_counter() - data_started
        inputs = cpu_inputs.to(DEVICE, dtype=torch.long, non_blocking=True)
        labels = cpu_labels.to(DEVICE, dtype=torch.long, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            output = train_model(inputs, labels=labels)
            loss = output.loss
        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"{spec.arm} step={step}: non-finite loss {loss}"
            )
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            raw_model.parameters(), 1.0, error_if_nonfinite=True
        )
        optimizer.step()

        effective = int(batch_metrics["predicted_tokens"])
        world_tokens = int(batch_metrics["world_predicted_tokens"])
        seen_tokens += effective
        seen_world_tokens += world_tokens
        step += 1
        interval_steps += 1
        interval_effective_tokens += effective
        interval_world_tokens += world_tokens
        interval_positions += MICRO_BATCH * (SEQ_LEN - 1)
        interval_losses.append(loss.detach())

        del inputs, labels, output, loss, cpu_inputs, cpu_labels

        if step % LOG_EVERY == 0:
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - interval_start
            mean_loss = float(torch.stack(interval_losses).mean())
            actual_ratio = interval_world_tokens / max(1, interval_effective_tokens)
            record = {
                "step": step,
                "seen_tokens": seen_tokens,
                "seen_world_tokens": seen_world_tokens,
                "loss": mean_loss,
                "lr": lr,
                "grad_norm": float(grad_norm),
                "effective_tokens_per_second": interval_effective_tokens / elapsed,
                "gpu_positions_per_second": interval_positions / elapsed,
                "world_token_ratio": actual_ratio,
                "data_wait_ms_per_step": 1000 * interval_data / max(1, interval_steps),
                "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
            }
            history.append(record)
            print(
                f"[{spec.arm}] step={step:05d} token={seen_tokens:,}/{spec.token_budget:,} "
                f"loss={mean_loss:.4f} lr={lr:.3e} grad={float(grad_norm):.3f} "
                f"eff={record['effective_tokens_per_second']:,.0f} tok/s "
                f"gpu={record['gpu_positions_per_second']:,.0f} pos/s "
                f"world={actual_ratio:.2%} data={record['data_wait_ms_per_step']:.1f}ms "
                f"peak={record['peak_vram_gib']:.2f}GiB"
            )
            interval_start = time.perf_counter()
            interval_effective_tokens = 0
            interval_positions = 0
            interval_world_tokens = 0
            interval_losses = []
            interval_data = 0.0
            interval_steps = 0

        should_validate = step % VALIDATE_EVERY == 0
        should_save = step % SAVE_EVERY == 0
        if should_validate:
            validation = evaluate_natural_loss(raw_model)
            print(
                f"[{spec.arm}] natural_val={validation['loss']:.6f} "
                f"ppl={validation['perplexity']:.2f}"
            )
        if should_save:
            save_training_checkpoint(
                latest_path,
                raw_model=raw_model,
                optimizer=optimizer,
                spec=spec,
                step=step,
                seen_tokens=seen_tokens,
                seen_world_tokens=seen_world_tokens,
                history=history,
                source=source,
            )
        if should_validate or should_save:
            torch.cuda.synchronize()
            interval_start = time.perf_counter()

    natural_validation = evaluate_natural_loss(raw_model)
    save_final_checkpoint(
        final_path,
        raw_model=raw_model,
        spec=spec,
        step=step,
        seen_tokens=seen_tokens,
        seen_world_tokens=seen_world_tokens,
        history=history,
    )
    result = {
        "spec": asdict(spec),
        "parameters": raw_model.num_trainable_parameters(),
        "step": step,
        "seen_tokens": seen_tokens,
        "seen_world_tokens": seen_world_tokens,
        "actual_world_ratio": seen_world_tokens / seen_tokens,
        "natural_validation": natural_validation,
        "wall_seconds": time.time() - started,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
        "history": history,
    }
    result_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if latest_path.exists():
        latest_path.unlink()
        print("The completed run's optimizer-containing latest.pt was removed; final.pt was preserved.")
    print(json.dumps({k: v for k, v in result.items() if k != "history"}, indent=2))

    del train_model, raw_model, optimizer, source, world_store
    clear_cuda()
    return final_path


## 19. Define WorldPairs evaluation metrics

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 19 — WorldPairs evaluation
def encode_candidate(prompt: str, candidate: str) -> dict:
    full_text = prompt + " " + candidate + "."
    encoding = tokenizer.encode(full_text, add_special_tokens=False)
    boundary = len(prompt)
    target_positions = [
        index
        for index, (_start, end) in enumerate(encoding.offsets)
        if end > boundary
    ]
    if not target_positions or min(target_positions) == 0:
        raise ValueError("Could not derive candidate token boundary")
    if len(encoding.ids) > 2_048:
        raise ValueError(
            f"Evaluation example {len(encoding.ids)} token; model limit 2048"
        )
    return {
        "ids": encoding.ids,
        "target_positions": target_positions,
    }


@torch.inference_mode()
def score_candidate_tasks(
    model: nn.Module,
    tasks: list[dict],
    batch_size: int = 12,
) -> list[dict]:
    prepared = []
    for task_id, task in enumerate(tasks):
        encoded = encode_candidate(task["prompt"], task["candidate"])
        prepared.append({"task_id": task_id, **task, **encoded})
    prepared.sort(key=lambda item: len(item["ids"]))
    results: list[Optional[dict]] = [None] * len(tasks)
    model.eval()

    for start in range(0, len(prepared), batch_size):
        chunk = prepared[start:start + batch_size]
        maximum = max(len(item["ids"]) for item in chunk)
        batch = torch.full(
            (len(chunk), maximum),
            PAD_ID,
            dtype=torch.long,
            device=DEVICE,
        )
        for row, item in enumerate(chunk):
            batch[row, :len(item["ids"])] = torch.tensor(
                item["ids"], dtype=torch.long, device=DEVICE
            )
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = model(batch).logits
        for row, item in enumerate(chunk):
            positions = item["target_positions"]
            predictor_positions = torch.tensor(
                [position - 1 for position in positions], device=DEVICE
            )
            targets = torch.tensor(
                [item["ids"][position] for position in positions],
                device=DEVICE,
            )
            selected = logits[row].index_select(0, predictor_positions).float()
            token_scores = F.log_softmax(selected, dim=-1).gather(
                1, targets[:, None]
            )[:, 0]
            raw_score = float(token_scores.sum())
            normalized_score = float(token_scores.mean())
            results[item["task_id"]] = {
                **{k: item[k] for k in (
                    "record_index", "member", "candidate_index", "candidate"
                )},
                "raw_score": raw_score,
                "normalized_score": normalized_score,
                "answer_tokens": len(positions),
            }
        del batch, logits
    assert all(result is not None for result in results)
    return [result for result in results if result is not None]


def summarize_pair_rows(pair_rows: list[dict]) -> list[dict]:
    groups: list[tuple[str, str, list[dict]]] = [("all", "all", pair_rows)]
    for split in sorted({row["split"] for row in pair_rows}):
        subset = [row for row in pair_rows if row["split"] == split]
        groups.append(("split", split, subset))
    for split, kind in sorted(
        {(row["split"], row["contrast_kind"]) for row in pair_rows}
    ):
        subset = [
            row for row in pair_rows
            if row["split"] == split and row["contrast_kind"] == kind
        ]
        groups.append(("cell", f"{split}/{kind}", subset))

    summaries = []
    for group_type, group, rows in groups:
        for metric in ("raw", "normalized"):
            a_correct = np.asarray([row[f"a_correct_{metric}"] for row in rows])
            b_correct = np.asarray([row[f"b_correct_{metric}"] for row in rows])
            pair_correct = a_correct & b_correct
            changed = np.asarray([row[f"prediction_changed_{metric}"] for row in rows])
            margins = [
                value
                for row in rows
                for value in (row[f"margin_a_{metric}"], row[f"margin_b_{metric}"])
            ]
            summaries.append({
                "group_type": group_type,
                "group": group,
                "metric": metric,
                "pairs": len(rows),
                "individual_accuracy": float(
                    (a_correct.sum() + b_correct.sum()) / (2 * len(rows))
                ),
                "pair_accuracy": float(pair_correct.mean()),
                "prediction_flip_rate": float(changed.mean()),
                "mean_correct_margin": float(np.mean(margins)),
            })
    return summaries


def evaluate_worldpairs(
    model: nn.Module,
    records: list[dict],
    batch_size: int = 12,
) -> tuple[list[dict], list[dict]]:
    tasks = []
    for record_index, record in enumerate(records):
        for member in ("a", "b"):
            for candidate_index, candidate in enumerate(record["candidate_texts"]):
                tasks.append({
                    "record_index": record_index,
                    "member": member,
                    "candidate_index": candidate_index,
                    "candidate": candidate,
                    "prompt": record[f"prompt_{member}"],
                })
    scores = score_candidate_tasks(model, tasks, batch_size=batch_size)
    by_record: dict[tuple[int, str], list[dict]] = defaultdict(list)
    for item in scores:
        by_record[(item["record_index"], item["member"])].append(item)

    pair_rows = []
    for index, record in enumerate(records):
        row = {
            "family_id": record["family_id"],
            "split": record["split"],
            "contrast_kind": record["contrast_kind"],
            "reasoning_profile": record["reasoning_profile"],
            "chain_depth": record["chain_depth"],
        }
        for member in ("a", "b"):
            options = sorted(
                by_record[(index, member)],
                key=lambda item: item["candidate_index"],
            )
            correct_id = record[f"answer_id_{member}"]
            correct_index = record["candidate_answer_ids"].index(correct_id)
            for metric in ("raw", "normalized"):
                key = f"{metric}_score"
                values = [item[key] for item in options]
                prediction = int(np.argmax(values))
                alternatives = [
                    value for option_index, value in enumerate(values)
                    if option_index != correct_index
                ]
                row[f"prediction_{member}_{metric}"] = prediction
                row[f"{member}_correct_{metric}"] = prediction == correct_index
                row[f"margin_{member}_{metric}"] = (
                    values[correct_index] - max(alternatives)
                )
        for metric in ("raw", "normalized"):
            row[f"prediction_changed_{metric}"] = (
                row[f"prediction_a_{metric}"] != row[f"prediction_b_{metric}"]
            )
            row[f"pair_correct_{metric}"] = (
                row[f"a_correct_{metric}"] and row[f"b_correct_{metric}"]
            )
        pair_rows.append(row)
    return pair_rows, summarize_pair_rows(pair_rows)


## 20. Evaluate a completed checkpoint

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 20 — Checkpoint evaluation
def macro_pair_score(summary_rows: list[dict], metric: str = "normalized") -> float:
    split_rows = [
        row for row in summary_rows
        if row["group_type"] == "split" and row["metric"] == metric
    ]
    assert {row["group"] for row in split_rows} == {
        "id", "ood_template", "ood_depth", "ood_kind"
    }
    return float(np.mean([row["pair_accuracy"] for row in split_rows]))


def evaluate_run(
    final_path: Path,
    suite_path: Path,
    suite_label: str,
    force: bool = False,
) -> dict:
    run_dir = final_path.parent
    evaluation_dir = run_dir / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    summary_path = evaluation_dir / f"{suite_label}.summary.json"
    predictions_path = evaluation_dir / f"{suite_label}.pairs.jsonl"
    if summary_path.exists() and predictions_path.exists() and not force:
        print("Existing evaluation loaded:", summary_path)
        return json.loads(summary_path.read_text(encoding="utf-8"))

    checkpoint = torch.load(final_path, map_location="cpu", weights_only=False)
    assert checkpoint.get("format") == "anka-w3-final-v1"
    seed_everything(int(checkpoint["spec"]["seed"]))
    model = build_c2_worldpairs()
    model.load_state_dict(checkpoint["model"])
    model = model.to(DEVICE)
    clear_cuda()
    torch.cuda.reset_peak_memory_stats()

    natural = evaluate_natural_loss(model)
    records = load_evaluation_suite(suite_path)
    started = time.perf_counter()
    pair_rows, world_summary = evaluate_worldpairs(model, records)
    torch.cuda.synchronize()
    world_seconds = time.perf_counter() - started
    result = {
        "arm": checkpoint["spec"]["arm"],
        "stage": checkpoint["spec"]["stage"],
        "seed": checkpoint["spec"]["seed"],
        "parameters": model.num_trainable_parameters(),
        "seen_tokens": checkpoint["seen_tokens"],
        "seen_world_tokens": checkpoint["seen_world_tokens"],
        "suite": suite_label,
        "suite_sha256": sha256_file(suite_path),
        "natural_validation": natural,
        "worldpairs_macro_pair_accuracy_raw": macro_pair_score(world_summary, "raw"),
        "worldpairs_macro_pair_accuracy_normalized": macro_pair_score(
            world_summary, "normalized"
        ),
        "worldpairs_seconds": world_seconds,
        "worldpairs_summary": world_summary,
        "peak_vram_gib": torch.cuda.max_memory_allocated() / 2**30,
    }
    with predictions_path.open("w", encoding="utf-8") as handle:
        for row in pair_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    summary_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps({k: v for k, v in result.items() if k != "worldpairs_summary"}, indent=2))
    display(pd.DataFrame(world_summary))
    del model, checkpoint
    clear_cuda()
    return result


## 21. Train and evaluate the three matched conditions

This is a compute-intensive stage. Confirm the selected runtime, input paths, token budget, and resume directory before execution. Checkpoints are written incrementally.


In [ ]:
# 21 — Train and evaluate three matched conditions
STAGE = "proof_128m_v1"
SPECS = [
    RunSpec(
        stage=STAGE,
        arm="D0_NATURAL",
        token_budget=TOKEN_BUDGET,
        seed=SEED,
        world_ratio=0.0,
        world_corpus=None,
        world_manifest=None,
    ),
    RunSpec(
        stage=STAGE,
        arm="D1_WRONG_PAIRS",
        token_budget=TOKEN_BUDGET,
        seed=SEED,
        world_ratio=WORLD_RATIO,
        world_corpus="matched_unpaired",
        world_manifest=str(WORLD_MANIFEST),
    ),
    RunSpec(
        stage=STAGE,
        arm="W3_CORRECT_PAIRS",
        token_budget=TOKEN_BUDGET,
        seed=SEED,
        world_ratio=WORLD_RATIO,
        world_corpus="paired",
        world_manifest=str(WORLD_MANIFEST),
    ),
]

FINAL_PATHS = {}
RESULTS = {}
for spec in SPECS:
    FINAL_PATHS[spec.arm] = train_run(spec)
for spec in SPECS:
    RESULTS[spec.arm] = evaluate_run(
        FINAL_PATHS[spec.arm],
        PROOF_SUITE,
        "proof",
    )


## 22. Build the final decision table and report

This section evaluates an existing checkpoint and saves machine-readable results. It should not change the released model weights.


In [ ]:
# 22 — Single result table and decision
def load_pair_predictions(final_path: Path) -> list[dict]:
    path = final_path.parent / "evaluation/proof.pairs.jsonl"
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


def paired_test(treatment_path: Path, control_path: Path) -> dict:
    treatment = {row["family_id"]: row for row in load_pair_predictions(treatment_path)}
    control = {row["family_id"]: row for row in load_pair_predictions(control_path)}
    assert treatment.keys() == control.keys()
    keys = sorted(treatment)
    t = np.asarray([bool(treatment[key]["pair_correct_normalized"]) for key in keys])
    c = np.asarray([bool(control[key]["pair_correct_normalized"]) for key in keys])
    treatment_only = int((t & ~c).sum())
    control_only = int((c & ~t).sum())
    discordant = treatment_only + control_only
    p_value = float(binomtest(treatment_only, discordant, 0.5).pvalue) if discordant else 1.0
    return {
        "difference": float(t.mean() - c.mean()),
        "p_value": p_value,
        "treatment_only": treatment_only,
        "control_only": control_only,
    }


rows = []
for spec in SPECS:
    result = RESULTS[spec.arm]
    training = json.loads((spec.run_dir / "training_result.json").read_text(encoding="utf-8"))
    rows.append({
        "arm": spec.arm,
        "natural_loss": float(result["natural_validation"]["loss"]),
        "world_pair_accuracy": float(result["worldpairs_macro_pair_accuracy_normalized"]),
        "world_pair_accuracy_raw": float(result["worldpairs_macro_pair_accuracy_raw"]),
        "actual_world_ratio": float(training["actual_world_ratio"]),
        "training_minutes": float(training["wall_seconds"]) / 60.0,
    })

table = pd.DataFrame(rows).sort_values("world_pair_accuracy", ascending=False).reset_index(drop=True)
display(table.style.format({
    "natural_loss": "{:.4f}",
    "world_pair_accuracy": "{:.2%}",
    "world_pair_accuracy_raw": "{:.2%}",
    "actual_world_ratio": "{:.2%}",
    "training_minutes": "{:.1f}",
}))

w3 = RESULTS["W3_CORRECT_PAIRS"]
d0 = RESULTS["D0_NATURAL"]
d1 = RESULTS["D1_WRONG_PAIRS"]
w3_score = float(w3["worldpairs_macro_pair_accuracy_normalized"])
d0_score = float(d0["worldpairs_macro_pair_accuracy_normalized"])
d1_score = float(d1["worldpairs_macro_pair_accuracy_normalized"])

stats_d0 = paired_test(FINAL_PATHS["W3_CORRECT_PAIRS"], FINAL_PATHS["D0_NATURAL"])
stats_d1 = paired_test(FINAL_PATHS["W3_CORRECT_PAIRS"], FINAL_PATHS["D1_WRONG_PAIRS"])
natural_delta = float(w3["natural_validation"]["loss"] - d0["natural_validation"]["loss"])

works = (
    w3_score > d0_score
    and w3_score > d1_score
    and stats_d0["p_value"] < 0.05
    and stats_d1["p_value"] < 0.05
)

report = {
    "verdict": "W3 WORKS" if works else "W3 NOT PROVEN",
    "token_budget_per_arm": int(TOKEN_BUDGET),
    "world_ratio": float(WORLD_RATIO),
    "results": rows,
    "w3_vs_d0": stats_d0,
    "w3_vs_d1": stats_d1,
    "w3_natural_loss_delta_vs_d0": natural_delta,
    "acceptance": {
        "w3_beats_d0": bool(w3_score > d0_score),
        "w3_beats_d1": bool(w3_score > d1_score),
        "w3_vs_d0_p_lt_0_05": bool(stats_d0["p_value"] < 0.05),
        "w3_vs_d1_p_lt_0_05": bool(stats_d1["p_value"] < 0.05),
        "natural_loss_delta_vs_d0": natural_delta,
    },
}

REPORT_JSON = WORLD_ROOT / "worldpairs_proof_result.json"
REPORT_CSV = WORLD_ROOT / "worldpairs_proof_result.csv"
REPORT_JSON.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
table.to_csv(REPORT_CSV, index=False)

print("\n" + "=" * 72)
print(report["verdict"])
print("=" * 72)
print(f"W3 - D0 pair-accuracy difference: {stats_d0['difference']:+.2%} | p={stats_d0['p_value']:.6g}")
print(f"W3 - D1 pair-accuracy difference: {stats_d1['difference']:+.2%} | p={stats_d1['p_value']:.6g}")
print(f"W3 - D0 natural-loss difference   : {natural_delta:+.4f}")
print("report:", REPORT_JSON)
